# Thermo RAW to three reader-friendly NetCDF levels
 Keep this notebook beside `corems_topn_processing.py`, or set CODE_DIR. Run in the existing CoreMS/Thermo raw kernel.

| Output | Contents | Chemical cleaning |
|---|---|---|
| a1 | One average of the selected sample scans: profile arrays and picked peaks | None |
| b1 | The same average, followed by the original RP filter, recalibration, and formula assignment | None |
| c1 | b1 unambiguous peaks: ratio-based background removal, optional internal-standard removal, oxygen filter, then DBE-O cleaning | Gap rule; without a reliable gap, remove DBE-O > 1 for CHOS/CHNOS |

Each completed level has a separate NetCDF file.
Selection diagnostics are also stored in NetCDF, even if sample selection fails. 
a1 to 2, peaks do not pass resovling power filter will be removed. 
c1 is skipped when background mode is `none` or no eligible background scans remain. A later failure does not invalidate completed earlier files.
b1 retains every unambiguous, ambiguous, and unassigned measured peak. Multiple candidate formulas for an ambiguous peak link to one measured peak through `formula_peak_id` → `peak_id`. c1 contains only the retained b1 unambiguous peaks.

The final c1 van Krevelen PNG is available both in `diagnostics/plots/07_level3_van_krevelen.png` and in the `van_krevelen_png` byte variable of the c1 `.c1.` NetCDF. Source metadata are in `data/raw_metadata.json` (and `data/background_raw_metadata.json` for an external RAW).
Internal-standard removal is an independent, optional c1 step. It does not normalize or subtract peak intensity. Check the configured formulas before switching it on. Processing revision 12 retains the existing public NetCDF schema.

Every public level now contains all sample metadata as global attributes. `raw_metadata.json` remains a convenient duplicate. Field sample dates/times do not replace the RAW acquisition `data_acquisition_date`.

Named outputs: prefix.a1.YYYYMMDD.HHMMSS.nc, prefix.b1.YYYYMMDD.HHMMSS.nc and prefix.c1.YYYYMMDD.HHMMSS.nc. The file naming codes map to the existing Levels 1, 2 and 3. They do not change scientific processing or assert ARM approval.

Public files identify their product with file_level_code = a1/b1/c1. They do not store output_level, new_file_name, description, peak_source or snr_definition as global attributes. RAW acquisition date is stored as data_acquisition_date and is independent of the filename date and field sampling dates. Tracer name and m/z are adjacent attributes. Existing files can be migrated with the matching 02 metadata_only mode. Internal workflow and diagnostic identifiers retain compatibility with saved runs.

Automatic DBE-O gaps now require a protected main distribution on their left. GAP_SETTINGS.min_left_fraction defaults to 0.8. Gap diagnostics record candidate support, retained fraction, mode position and rejection reasons. This is a processing rule change; metadata_only cannot recompute c1.


In [ ]:
# Install only if missing, in the same CoreMS environment.
# %pip install netCDF4


## 1. Imports and paths

In [ ]:
from pathlib import Path
from dataclasses import asdict
from datetime import datetime, timezone
import hashlib
import json
import re
import sys
import uuid

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from netCDF4 import Dataset

CODE_DIR = Path(r"C:\Users\nius206\Documents\ARM_nanodesi")
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
import corems_topn_processing as topn
from corems_topn_processing import CoreMSConfig, BackgroundConfig

RAW_FILE = Path(r"C:\Users\nius206\Documents\2026Developer\raw\BNF\x_EMSL61150_0pt25umOS_B2_BNFM1_03XX25_07N_1.raw")
OUTPUT_ROOT = Path(r"C:\Users\nius206\Documents\2026Developer\test\output_three_levels")
print("Processing module:", topn.__file__)

## 2. User settings

A later tracer is used only if every earlier tracer fails the same region criteria. For every
file with positive-TIC MS1 scans, Otsu splits the log1p of positive tracer EIC values;
zero EIC values remain low readings in the full trace. Bridge internal gaps
of at most ten scans. A candidate interval must contain at least 25 scans, have
at least 25% above threshold, and its median intensity must be at least
five times the low-state median when the latter is positive. There must be at least
ten below-threshold readings somewhere in the file to define a low reference.

Average every scan in the ROI if contains at most 50 scans. Otherwise
select the 50 highest *original* EIC values inside the intervals, breaking ties by
scan number. 
A raw file run without an accepted interval for any tracer raises an explicit error.
With `external_file` or `none`, use every eligible positive-TIC MS1 scan when there are
at most 50; otherwise draw 50 uniformly at random without replacement using a fixed
sample seed. These modes do not require or choose a tracer or a sample ROI.

Same-file background: rank positive-TIC scans outside *all* accepted sample intervals
by ascending EIC intensity of the *chosen* tracer and take at most 25. Sample intervals
include any interior low gaps, preventing these dips from becoming background.
External background: uniformly sample at most 25 eligible MS1 scans.
Zero eligible background scans skip c1.
Background validation and extraction happen only after b1 succeeds.

Gap detection uses unique formulas with DBE and total oxygen count. For each class,
search adjacent distinct DBE-O values for an interval at least 3 units wide and at least
3 times the median positive interval, with at least 2 formulas on each side and at least
10 formulas overall. Only consider gaps with at least 80% of unique formulas on the retained (left) side and all dominant modes on that side. Select the widest qualifying interval; ties select the higher cutoff. The retained-fraction setting is configurable.
The gap cutoff uses only originally unambiguous sample peaks left after background and optional internal-standard removal.
Optional internal-standard removal uses C/H/O/N/S composition to filter c1 unambiguous peaks after background removal and before oxygen/DBE-O cleaning. The setting starts off; first confirm that the three older formulas match this batch's internal standards. A sample isomer with the same elemental formula cannot be distinguished by this rule. b1 keeps all assignments, and `background_mode=none` still skips c1.


In [ ]:
# Selection settings: adjust these here, then rerun from this cell.
CANDIDATE_MODULES = {
    "sample_selection": {
        "MBTCA_C8H12O6": 203.05611,
        "2MTOS_C5H12O7S1": 215.02310,
        "C8H10O6": 201.04046,
        "nitrocatechol_C6H5NO4": 154.01458,
    },
}
TRACER_PRIORITY = ("MBTCA_C8H12O6", "2MTOS_C5H12O7S1", "C8H10O6", "nitrocatechol_C6H5NO4")
TOP_N_SAMPLE_SCANS = 50
SAMPLE_RANDOM_SEED = 42
MS_FILTER = "MS !d"
EIC_TOLERANCE_PPM = 10.0
ROI_SETTINGS = {
    "max_interior_gap_scans": 10,
    "min_region_scans": 25,
    "min_high_fraction": 0.25,
    "min_inside_to_low_fold": 1.5,
    "min_low_reference_scans": 10,
}
BACKGROUND_MODE = "same_file"  # same_file | external_file | none
BACKGROUND_RAW_PATH = None     # Path(r"C:\path\to\background.raw") for external_file
TOP_N_BACKGROUND_SCANS = 25
BACKGROUND_RANDOM_SEED = 42
BACKGROUND_REMOVAL_MODE = "ratio_guarded_raw_intensity"
SAMPLE_TO_BACKGROUND_RATIO_CUTOFF = 10.0

# Optional, independent of intensity normalization. Check these three legacy
# formulas against the actual standards used for this batch before enabling.
# Matching uses elemental composition, so an ambient isomer with the same
# formula would also be removed. No peak heights are changed.
REMOVE_INTERNAL_STANDARD_PEAKS = True
INTERNAL_STANDARD_FORMULAS = (
    "C6 H5 O6 N1 S1",
    "C7 H8 O4 S1",
    "C10 H8 O6 S1",
    "C12 H26 O4 S1",
)

GAP_SETTINGS = {
    # Count unique formulas, not intensity. Protect the main distribution.
    "min_left_fraction": 0.8,
    "min_formulas": 10,
    "min_formulas_each_side": 2,
    "min_width": 3.0,
    "min_spacing_ratio": 3.0,
    "round_decimals": 8,
}

# The first ten fields come from the sample RAW. The remaining fields describe processing.
RAW_ACQUISITION_METADATA_FIELDS = [
    "date", "instrument_type", "ion_mode", "ion_source", "mz_range",
    "agc", "ft_resolution", "max_injection_time", "agc_target", "n_ms1_scans",
]
PROCESSING_METADATA_FIELDS = ["selected_tracer_name", "selected_tracer_mz",
                              "n_sample_scans_averaged", "n_background_scans_averaged"]
METADATA_FIELDS = [*RAW_ACQUISITION_METADATA_FIELDS, *PROCESSING_METADATA_FIELDS]
# Scalars mean one consistently available value; lists preserve multiple values/missingness.
# None means unavailable. Date retains the RAW acquisition timestamp.
# Maximum injection time is stored in ms.

NETCDF_COMPRESSION_LEVEL = 4
REFERENCE_MASS_LIST_PATH = Path(r"C:\Users\nius206\Documents\nanodesi\Hawkes_neg 1.ref")
FORMULA_DATABASE_URL = "sqlite:///C:/Users/nius206/Documents/nanodesi/corems_mf.db"
corems_config = CoreMSConfig(
    reference_mass_list_path=str(REFERENCE_MASS_LIST_PATH),
    formula_database_url=FORMULA_DATABASE_URL,
    noise_threshold_log_nsigma=100.0,
    used_atoms={"C": (1, 90), "H": (1, 200), "O": (0, 20), "N": (0, 2), "S": (0, 1)},
)

# Field sample information from Excel, separate from the RAW acquisition date.
# Public products use a1/b1/c1. Internal workflow keys remain compatible with saved runs.
REMOVED_PUBLIC_GLOBAL_ATTRIBUTES = (
    "schema_version", "metadata_global_attributes_version", "field_time_zone",
    "metadata_value_encoding", "description", "peak_source", "snr_definition",
    "output_level", "new_file_name", "date","n_formula_rows",  "min_altitude_units",
    "max_altitude_units",
)
FIELD_METADATA_COLUMNS = {
    "Location": "location",
    "Site": "site",
    "System": "system",
    "Min. Altitude (m)": "min_altitude_m",
    "Max. Altitude (m)": "max_altitude_m",
    "Start Day": "start_day",
    "Start Time (UTC)": "start_time_utc",
    "End Day": "end_day",
    "End Time (UTC)": "end_time_utc",
    "Field Note": "field_note",
}
# For a single RAW run, fill these values here. Batch jobs override them per row.
FIELD_METADATA = {name: None if name.endswith("_altitude_m") else ""
                  for name in FIELD_METADATA_COLUMNS.values()}

# Required for a single RAW run. Batch supplies this from Excel.
NEW_FILE_NAME = "bnfhrmstbac250m500mS4.20250530.120507"  # prefix.YYYYMMDD.HHMMSS, without a level code



## 3. Reading, scoring, metadata, and export helpers

This cell defines functions. It reuses the original averaging, RP filtering, calibration, formula-search configuration, assignment classification, and background-removal functions.

In [ ]:
def json_text(value):
    return json.dumps(value, ensure_ascii=False, indent=2, default=topn._json_default)


def save_json(path, value):
    Path(path).write_text(json_text(value), encoding="utf-8")


def nc_text(group, name, value):
    group.createVariable(name, str, ())[...] = str(value)


def write_nc_table(parent, name, frame):
    """Save table columns with original names, types, and explicit validity masks."""
    group = parent.createGroup(name)
    group.createDimension("row", len(frame) or None)
    group.row_count = len(frame)
    group.columns_json = json_text([str(c) for c in frame.columns])
    for i, column in enumerate(frame.columns):
        series = frame[column]
        key = f"c{i:04d}"
        missing = series.isna().to_numpy()
        kwargs = dict(zlib=True, complevel=NETCDF_COMPRESSION_LEVEL)
        if pd.api.types.is_bool_dtype(series.dtype):
            kind, values = "bool", series.fillna(False).to_numpy(dtype=np.uint8)
            variable = group.createVariable(key, "u1", ("row",), **kwargs)
        elif pd.api.types.is_integer_dtype(series.dtype):
            unsigned = pd.api.types.is_unsigned_integer_dtype(series.dtype)
            kind = "integer"
            values = series.fillna(0).to_numpy(dtype=np.uint64 if unsigned else np.int64)
            variable = group.createVariable(key, "u8" if unsigned else "i8", ("row",), **kwargs)
        elif pd.api.types.is_numeric_dtype(series.dtype):
            kind, values = "float", series.to_numpy(dtype=float, na_value=np.nan)
            variable = group.createVariable(key, "f8", ("row",), **kwargs)
        else:
            kind = "string"
            values = np.asarray(["" if absent else str(v) for v, absent in zip(series, missing)], dtype=object)
            variable = group.createVariable(key, str, ("row",))
        variable.original_column = str(column)
        variable.original_dtype = str(series.dtype)
        variable.logical_type = kind
        valid = group.createVariable(key + "_valid", "u1", ("row",), **kwargs)
        if len(frame):
            variable[:] = values
            valid[:] = (~missing).astype(np.uint8)
    return group


def read_nc_table(group):
    data = {}
    for i, column in enumerate(json.loads(group.columns_json)):
        key = f"c{i:04d}"
        variable = group.variables[key]
        values = np.ma.getdata(variable[:])
        valid = np.asarray(group.variables[key + "_valid"][:], dtype=bool)
        kind = variable.logical_type
        if kind == "bool":
            series = pd.Series(values.astype(bool), dtype="boolean" if not valid.all() else bool)
        elif kind == "integer" and not valid.all():
            series = pd.Series(values, dtype="UInt64" if values.dtype.kind == "u" else "Int64")
        else:
            series = pd.Series(values, dtype=object if kind == "string" else None)
        if not valid.all():
            series.loc[~valid] = pd.NA if kind in {"bool", "integer", "string"} else np.nan
        data[column] = series
    return pd.DataFrame(data)


def as_float(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return np.nan


def positive_or_nan(value):
    value = as_float(value)
    return value if np.isfinite(value) and value > 0 else np.nan


def optional(call):
    try:
        return call()
    except Exception:
        return None


def candidate_rows(modules):
    return [(module, name, float(mz)) for module, ions in modules.items() for name, mz in ions.items()]


def extract_scan_signals(raw_path, modules, ms_filter, tolerance_ppm):
    """Read TIC and optional EICs. Missing EIC entries remain missing, not invented zeros."""
    reader = topn._import_corems()["ThermoBaseClass"](str(raw_path))
    try:
        reader.chromatogram_settings.scans = (-1, -1)
        reader.chromatogram_settings.eic_tolerance_ppm = float(tolerance_ppm)
        tic, _ = reader.get_tic(ms_type=ms_filter, peak_detection=False, smooth=False,
                                plot=False, trace_type="TIC")
        if tic is None or len(tic.scans) == 0:
            return pd.DataFrame(columns=["scan", "time_min", "tic"])
        frame = pd.DataFrame({"scan": np.asarray(tic.scans, dtype=int),
                              "time_min": np.asarray(tic.time, dtype=float),
                              "tic": np.asarray(tic.tic, dtype=float)})
        if frame["scan"].duplicated().any():
            raise ValueError("Duplicate scan IDs returned by the Thermo chromatogram API")
        targets = candidate_rows(modules)
        if targets:
            eics, _ = reader.get_eics(target_mzs=[r[2] for r in targets], tic_data=tic,
                                      ms_type=ms_filter, peak_detection=False, smooth=False, plot=False)
            for _, name, mz in targets:
                try:
                    eic = topn._closest_eic(eics, mz, tolerance_ppm)
                except (KeyError, ValueError):
                    frame[name] = np.nan
                    continue
                values = pd.DataFrame({"scan": np.asarray(eic.scans, dtype=int),
                                       name: np.asarray(eic.eic, dtype=float)})
                if values["scan"].duplicated().any():
                    raise ValueError(f"Duplicate EIC scan IDs for {name}")
                frame = frame.merge(values, on="scan", how="left", validate="one_to_one")
        return frame.sort_values("scan").reset_index(drop=True)
    finally:
        topn._close_reader(reader)


def otsu_positive_cutoff(intensities, bins=256):
    """Fit Otsu on strictly positive log1p(EIC) values; return the raw EIC cutoff."""
    x = np.asarray(intensities, dtype=float)
    transformed = np.log1p(x[x > 0])
    if len(transformed) < 2 or np.ptp(transformed) <= np.finfo(float).eps:
        return np.nan
    histogram, edges = np.histogram(transformed, bins=bins)
    probability = histogram.astype(float) / histogram.sum()
    middle = (edges[:-1] + edges[1:]) / 2
    weight = np.cumsum(probability)
    weighted_mean = np.cumsum(probability * middle)
    denominator = weight * (1.0 - weight)
    variance = np.full(bins, -np.inf)
    allowed = denominator > 0
    variance[allowed] = (weighted_mean[-1] * weight[allowed] - weighted_mean[allowed]) ** 2 / denominator[allowed]
    if not np.isfinite(variance).any():
        return np.nan
    return float(np.expm1(edges[np.argmax(variance) + 1]))


def fill_short_interior_gaps(high, max_gap):
    filled = np.asarray(high, dtype=bool).copy()
    positions = np.flatnonzero(filled)
    for left, right in zip(positions[:-1], positions[1:]):
        if 0 < right - left - 1 <= max_gap:
            filled[left + 1:right] = True
    return filled


def contiguous_regions(mask):
    changes = np.diff(np.r_[False, np.asarray(mask, dtype=bool), False].astype(int))
    return list(zip(np.flatnonzero(changes == 1), np.flatnonzero(changes == -1)))


def analyze_eic(intensities, settings):
    """Return accepted sample ROI positions; low EIC values are not designated background."""
    x = np.asarray(intensities, dtype=float)
    n = len(x)
    empty = np.zeros(n, dtype=bool)
    result = {"method": "otsu_positive", "cutoff": np.nan,
              "n_reference_low": 0, "n_raw_high": 0, "regions": [],
              "mask": empty.copy(), "high_mask": empty.copy(),
              "status": "no_candidate", "reason": "", "region_details": []}
    if n == 0 or not np.isfinite(x).all() or (x < 0).any():
        result.update(status="unavailable", reason="missing_or_invalid_eic")
        return result
    if np.ptp(x) == 0:
        result.update(reason="flat_eic_no_separable_region")
        return result
    cutoff = otsu_positive_cutoff(x)
    if not np.isfinite(cutoff):
        result.update(reason="no_finite_positive_only_otsu_cutoff")
        return result
    high = x > cutoff
    low = x[~high]
    result.update(cutoff=cutoff, high_mask=high,
                  n_raw_high=int(high.sum()), n_reference_low=int(len(low)))
    if len(low) < settings["min_low_reference_scans"]:
        result.update(reason="no_adequate_low_state_in_this_file")
        return result
    low_reference = float(np.median(low))
    bridged = contiguous_regions(fill_short_interior_gaps(high, settings["max_interior_gap_scans"]))
    accepted = empty.copy()
    details = []
    for i, (start, stop) in enumerate(bridged):
        low_before = start - (bridged[i - 1][1] if i else 0)
        low_after = (bridged[i + 1][0] if i + 1 < len(bridged) else n) - stop
        inside_median = float(np.median(x[start:stop]))
        density = float(high[start:stop].mean())
        fold = inside_median / low_reference if low_reference > 0 else None
        contrast = fold >= settings["min_inside_to_low_fold"] if fold is not None else inside_median > 0
        passed = (stop - start >= settings["min_region_scans"]
                  and density >= settings["min_high_fraction"]
                  and contrast)
        details.append({"first_index": int(start), "stop_index": int(stop),
                        "n_scans": int(stop - start), "raw_high_fraction": density,
                        "n_low_scans_before": int(low_before), "n_low_scans_after": int(low_after),
                        "touches_file_start": bool(start == 0), "touches_file_end": bool(stop == n),
                        "median_intensity": inside_median, "low_reference_median": low_reference,
                        "inside_to_low_fold": float(fold) if fold is not None else None,
                        "contrast_check": "finite_fold" if fold is not None else "positive_over_zero_reference",
                        "passed": bool(passed)})
        if passed:
            accepted[start:stop] = True
    result.update(mask=accepted, regions=contiguous_regions(accepted), region_details=details)
    result.update(status="candidate" if accepted.any() else "no_candidate",
                  reason="diagnostic_region_found" if accepted.any()
                  else "no_region_met_duration_occupancy_and_contrast_checks")
    return result


def select_sample_scans(frame, top_n, analysis, tracer_name=None):
    if frame.empty:
        raise ValueError("No positive-TIC MS1 scans were returned. Check RAW readability, controller, and MS_FILTER.")
    if analysis is None or analysis["status"] != "candidate" or not np.any(analysis["mask"]):
        reason = analysis["reason"] if analysis is not None else "no_accepted_tracer"
        raise ValueError(f"No applicable sample region for {tracer_name}: {reason}")
    qualified = frame.loc[analysis["mask"]].copy()
    intensity = pd.to_numeric(qualified[tracer_name], errors="coerce")
    if not np.isfinite(intensity).all():
        raise ValueError(f"The accepted sample region contains missing or invalid {tracer_name} EIC values")
    if len(qualified) <= top_n:
        selected = qualified.sort_values("scan").copy()
        selected["selection_rank"] = np.arange(1, len(selected) + 1)
        selected["selection_rule"] = "all_scans_in_accepted_roi_at_or_below_limit"
    else:
        selected = qualified.sort_values([tracer_name, "scan"], ascending=[False, True]).head(top_n).copy()
        selected["selection_rank"] = np.arange(1, len(selected) + 1)
        selected["selection_rule"] = "top_raw_chosen_tracer_eic_within_accepted_roi"
    return selected.sort_values("scan").reset_index(drop=True)


def select_unrestricted_sample_scans(frame, top_n, seed):
    """Use every eligible scan up to the limit, otherwise uniformly draw a fixed-size sample."""
    if frame.empty:
        raise ValueError("No positive-TIC MS1 scans were returned from the sample RAW")
    candidates = frame.sort_values("scan").reset_index(drop=True)
    if len(candidates) <= top_n:
        selected = candidates.copy()
        rule = "all_eligible_ms1_scans_at_or_below_limit"
    else:
        indices = np.random.default_rng(seed).choice(len(candidates), size=top_n, replace=False)
        selected = candidates.iloc[indices].copy()
        rule = "uniform_random_eligible_ms1_scans_without_replacement"
    selected["selection_rank"] = np.arange(1, len(selected) + 1)
    selected["selection_rule"] = rule
    return selected.sort_values("scan").reset_index(drop=True)


def select_background_scans(frame, sample_scans, mode, limit, seed, roi_mask=None, tracer_name=None):
    candidates = frame.copy()
    if mode == "none":
        return candidates.iloc[:0].copy()
    valid = np.isfinite(pd.to_numeric(candidates["tic"], errors="coerce")) & candidates["tic"].gt(0)
    candidates = candidates.loc[valid].copy()
    if mode == "same_file":
        if roi_mask is None or len(roi_mask) != len(frame):
            raise ValueError("Same-file background needs an ROI mask for the full candidate table")
        candidates = candidates.loc[~pd.Series(roi_mask, index=frame.index).loc[candidates.index].astype(bool)]
        candidates = candidates.loc[~candidates["scan"].isin(sample_scans)]
        if candidates.empty:
            selected = candidates.copy()
        else:
            if not tracer_name or tracer_name not in candidates:
                raise ValueError("Same-file background requires the selected sample tracer EIC")
            candidates = candidates.loc[np.isfinite(pd.to_numeric(candidates[tracer_name], errors="coerce"))]
            selected = candidates.sort_values([tracer_name, "scan"]).head(limit).copy()
        selected["selection_rule"] = "lowest_raw_chosen_tracer_eic_outside_accepted_roi"
    elif mode == "external_file":
        candidates = candidates.sort_values("scan").reset_index(drop=True)
        indices = np.random.default_rng(seed).choice(len(candidates), size=min(limit, len(candidates)), replace=False)
        selected = candidates.iloc[indices].copy()
        selected["selection_rule"] = "uniform_random_without_replacement"
    else:
        raise ValueError("BACKGROUND_MODE must be same_file, external_file, or none")
    selected["selection_rank"] = np.arange(1, len(selected) + 1)
    return selected.sort_values("scan").reset_index(drop=True)


def metadata_label(value):
    return re.sub(r"[^a-z0-9]", "", str(value).lower())


METADATA_ALIASES = {
    "agc": ["AGC", "AGC On", "AGC Enabled", "AGC Mode", "AGC Status", "Automatic Gain Control"],
    "ft_resolution": ["FT Resolution", "Orbitrap Resolution"],
    "max_injection_time": ["Max Ion Time", "Max Ion Time (ms)", "Maximum Ion Time", "Max Ion Injection Time",
                           "Max Ion Injection Time (ms)", "Maximum Ion Injection Time (ms)", "Max Injection Time",
                           "Max Injection Time (ms)", "Maximum Injection Time (ms)", "Max IT (ms)"],
    "agc_target": ["AGC Target", "AGC Target Value", "AGC Target (ions)"],
}


def unique_value(values):
    unique = []
    for value in values:
        if value not in unique:
            unique.append(value)
    return unique[0] if len(unique) == 1 else unique or None


def readable_time(value):
    if value is None:
        return None
    if isinstance(value, datetime):
        return value.isoformat()
    converted = optional(lambda: value.ToString("o"))
    return str(converted if converted is not None else value)


def ms_order_number(filter_object, text):
    order = str(getattr(filter_object, "MSOrder", "")).lower()
    if order in {"ms", "ms1", "1"}:
        return 1
    found = re.fullmatch(r"ms(\d+)", order)
    if found:
        return int(found.group(1))
    found = re.search(r"\bms(\d*)\b", text, flags=re.IGNORECASE)
    if found:
        return int(found.group(1) or 1)
    raise ValueError("Cannot determine MS order; refusing to report an incorrect MS1 scan count")


def read_raw_metadata(raw_path, wanted):
    """Return only the requested acquisition fields, summarized over every MS1 scan."""
    reader = topn._import_corems()["ImportMassSpectraThermoMSFileReader"](str(raw_path))
    try:
        raw = reader.iRawDataPlus
        creation = optional(lambda: reader.iFileHeader.CreationDate)
        if creation is None:
            creation = optional(lambda: raw.CreationDate)
        instrument = optional(raw.GetInstrumentData)
        model = getattr(instrument, "Model", None)
        values = {name: [] for name in ["ion_mode", "ion_source", "mz_range", *METADATA_ALIASES]}
        count = 0
        for scan in range(int(raw.RunHeaderEx.FirstSpectrum), int(raw.RunHeaderEx.LastSpectrum) + 1):
            filt = optional(lambda: raw.GetFilterForScanNumber(scan))
            text = optional(lambda: raw.GetScanEventStringForScanNumber(scan)) or str(filt or "")
            if ms_order_number(filt, text) != 1:
                continue
            count += 1
            polarity_value = optional(lambda: filt.Polarity)
            polarity = str(polarity_value) if polarity_value is not None else None
            if polarity is None:
                token = re.search(r"(?:^|\s)([+-])(?:\s|$)", text)
                polarity = ("Positive" if token.group(1) == "+" else "Negative") if token else None
            source_value = optional(lambda: filt.IonizationMode)
            source = str(source_value) if source_value is not None else None
            if source is None:
                token = re.search(r"\b(NSI|ESI|APCI|APPI|MALDI|EI|CI)\b", text, re.IGNORECASE)
                source = token.group(1).upper() if token else None
            stats = optional(lambda: raw.GetScanStatsForScanNumber(scan))
            lo, hi = as_float(getattr(stats, "LowMass", None)), as_float(getattr(stats, "HighMass", None))
            ranges = re.findall(r"\[([^\]]+)\]", text)
            mz_range = f"{lo:g}-{hi:g} Th" if np.isfinite([lo, hi]).all() and 0 < lo <= hi else " | ".join(ranges) or None
            for name, value in [("ion_mode", polarity), ("ion_source", source), ("mz_range", mz_range)]:
                values[name].append(value)
            trailer = optional(lambda: raw.GetTrailerExtraInformation(scan))
            if trailer is not None:
                pairs = list(zip(trailer.Labels, trailer.Values))
            else:
                pairs = list((optional(lambda: reader.get_scan_header(scan)) or {}).items())
            for name, aliases in METADATA_ALIASES.items():
                labels = {metadata_label(label) for label in aliases}
                matches = [str(v).strip() or None for k, v in pairs if metadata_label(k) in labels]
                converted = []
                for value in matches or [None]:
                    if name == "agc" and value is not None:
                        value = {"on": "On", "off": "Off", "true": "On", "false": "Off", "1": "On", "0": "Off",
                                 "enabled": "On", "disabled": "Off", "yes": "On", "no": "Off"}.get(value.lower(), value)
                    elif name == "max_injection_time" and value is not None and np.isfinite(as_float(value)):
                        value = f"{float(value):g} ms"
                    elif value is not None and np.isfinite(as_float(value)):
                        value = float(value)
                        value = int(value) if value.is_integer() else value
                    converted.append(value)
                values[name].extend(converted)
        summary = {"date": readable_time(creation), "instrument_type": str(model) if model is not None else None,
                   **{name: unique_value(items) for name, items in values.items()}, "n_ms1_scans": count}
        unknown = set(wanted) - set(summary)
        if unknown:
            raise ValueError(f"Unknown metadata fields: {sorted(unknown)}")
        return {name: summary[name] for name in wanted}
    finally:
        topn._close_reader(reader)


def normalize_output_stem(value):
    """Validate Excel New File Name: prefix.YYYYMMDD.HHMMSS, without a level code."""
    value = str(value or '').strip()
    if value.lower().endswith('.nc'):
        value = value[:-3]
    match = re.fullmatch(r'([A-Za-z0-9][A-Za-z0-9_.-]*)\.(\d{8})\.(\d{6})', value)
    if not match or len(value) > 230:
        raise ValueError('New File Name must be prefix.YYYYMMDD.HHMMSS (no folder or level code)')
    prefix, day, clock = match.groups()
    if prefix.split('.')[-1].lower() in {'a1', 'b1', 'c1'}:
        raise ValueError('New File Name must omit the a1/b1/c1 level code')
    if prefix.split('.')[0].upper() in {'CON', 'PRN', 'AUX', 'NUL', *{f'COM{i}' for i in range(1,10)}, *{f'LPT{i}' for i in range(1,10)}}:
        raise ValueError('New File Name uses a reserved Windows filename')
    datetime.strptime(day + clock, '%Y%m%d%H%M%S')
    return value


def public_level_filename(new_file_name, level):
    """Insert the requested file level code before the date and UTC time fields."""
    codes = {'level1': 'a1', 'level2': 'b1', 'level3': 'c1'}
    if level not in codes:
        raise ValueError(f'Unknown public output level: {level}')
    prefix, day, clock = normalize_output_stem(new_file_name).rsplit('.', 2)
    return f'{prefix}.{codes[level]}.{day}.{clock}.nc'


def public_product_code(nc):
    """Read a1/b1/c1; accept legacy output_level and reject conflicting identifiers."""
    legacy_codes = {"level1": "a1", "level2": "b1", "level3": "c1"}
    code = str(getattr(nc, "file_level_code", "")).strip()
    legacy = str(getattr(nc, "output_level", "")).strip()
    if code and code not in legacy_codes.values():
        raise ValueError(f"Unknown file_level_code: {code}")
    if legacy and legacy not in legacy_codes:
        raise ValueError(f"Unknown legacy output_level: {legacy}")
    if code and legacy and code != legacy_codes[legacy]:
        raise ValueError("Conflicting file_level_code and legacy output_level")
    if code:
        return code
    if legacy:
        return legacy_codes[legacy]
    raise ValueError("Missing file_level_code (a1/b1/c1)")


def public_output_level(nc):
    """Translate to existing internal workflow keys without changing saved ledgers."""
    return {"a1": "level1", "b1": "level2", "c1": "level3"}[public_product_code(nc)]


def normalize_public_metadata(metadata):
    """Rename only the acquisition timestamp key; keep its original value."""
    result = dict(metadata)
    if "date" in result:
        old_date = result.pop("date")
        if "data_acquisition_date" in result and json.dumps(result["data_acquisition_date"], default=str) != json.dumps(old_date, default=str):
            raise ValueError("Conflicting date and data_acquisition_date metadata")
        result["data_acquisition_date"] = old_date
    return {name: value for name, value in result.items()
            if name not in REMOVED_PUBLIC_GLOBAL_ATTRIBUTES}


def rename_public_level_files(state, new_file_name):
    """Rename completed levels and persist their paths. Refuse existing targets."""
    import copy
    new_file_name = normalize_output_stem(new_file_name)
    data_dir = (Path(state['run_directory']) / 'data').resolve()
    original_state = copy.deepcopy(state)
    plan = []
    for level, path_string in state.get('netcdf_files', {}).items():
        if state.get('levels', {}).get(level, {}).get('status') != 'success':
            continue
        old = Path(path_string).resolve()
        new = data_dir / public_level_filename(new_file_name, level)
        if old.parent != data_dir or new.resolve().parent != data_dir:
            raise ValueError('Public NetCDF paths must remain in this run\'s data folder')
        if not old.is_file():
            raise FileNotFoundError(old)
        same_path = str(old).casefold() == str(new).casefold()
        if not same_path and new.exists():
            raise FileExistsError(f'Refusing to replace existing NetCDF: {new}')
        with Dataset(old) as nc:
            if public_output_level(nc) != level or nc.processing_status != 'success':
                raise ValueError(f'Unexpected saved level/status: {old}')
            previous = {key: nc.getncattr(key) for key in ('output_filename','file_level_code') if key in nc.ncattrs()}
        plan.append((level, old, old if same_path else new, previous))
    moved, changed_attrs = [], []
    try:
        for level, old, new, previous in plan:
            if old != new:
                if new.exists():
                    raise FileExistsError(new)
                old.rename(new)
                moved.append((old, new))
            changed_attrs.append((new, previous))
            with Dataset(new, 'a') as nc:
                nc.output_filename = new.name
                nc.file_level_code = {'level1':'a1','level2':'b1','level3':'c1'}[level]
            state['netcdf_files'][level] = str(new)
            state['levels'][level]['netcdf'] = str(new)
        state_path = Path(state['run_directory']) / 'diagnostics' / 'three_level_status.json'
        temporary = state_path.with_name(state_path.name + '.metadata.tmp')
        save_json(temporary, state)
        temporary.replace(state_path)
    except BaseException:
        for path, previous in reversed(changed_attrs):
            with Dataset(path, 'a') as nc:
                for key in ('output_filename','file_level_code'):
                    if key in previous:
                        nc.setncattr(key, previous[key])
                    elif key in nc.ncattrs():
                        nc.delncattr(key)
        for old, new in reversed(moved):
            new.rename(old)
        state.clear()
        state.update(original_state)
        raise


def normalized_field_metadata(values):
    """Ten field metadata values, kept distinct from RAW acquisition metadata."""
    from datetime import date, datetime, time
    values = dict(values or {})
    allowed = set(FIELD_METADATA_COLUMNS.values())
    unknown = set(values) - allowed
    if unknown:
        raise ValueError(f"Unknown field metadata: {sorted(unknown)}")
    result = {}
    for name in FIELD_METADATA_COLUMNS.values():
        raw = values.get(name)
        missing = raw is None or (isinstance(raw, str) and not raw.strip()) or pd.isna(raw)
        if missing:
            result[name] = None if name.endswith('_altitude_m') else ''
            continue
        if name.endswith('_altitude_m'):
            value = float(raw)
            if not np.isfinite(value):
                raise ValueError(f"{name} must be a finite number in meters")
            result[name] = value
        elif name.endswith('_day'):
            if isinstance(raw, datetime):
                if raw.time() != time(0):
                    raise ValueError(f"{name}: put the time in its separate UTC time column")
                value = raw.date().isoformat()
            elif isinstance(raw, date):
                value = raw.isoformat()
            else:
                value = str(raw).strip()
                # pandas dtype=str represents Excel date cells this way.
                value = re.sub(r'[ T]00:00:00(?:\.0+)?$', '', value)
                if not re.fullmatch(r'\d{4}-\d{2}-\d{2}', value):
                    raise ValueError(f"{name} must be a full date YYYY-MM-DD")
                value = date.fromisoformat(value).isoformat()
            result[name] = value
        elif name.endswith('_time_utc'):
            value = str(raw).strip()
            if not re.fullmatch(r'\d{2}:\d{2}(?::\d{2}(?:\.\d{1,6})?)?', value):
                raise ValueError(f"{name} must be UTC HH:MM or HH:MM:SS (Excel time cells are supported)")
            value = time.fromisoformat(value)
            result[name] = value.isoformat(timespec='microseconds' if value.microsecond else 'seconds')
        else:
            result[name] = str(raw).strip()
    if result['min_altitude_m'] is not None and result['max_altitude_m'] is not None and result['min_altitude_m'] > result['max_altitude_m']:
        raise ValueError('Min. Altitude (m) must not exceed Max. Altitude (m)')
    if all(result.get(k) for k in ('start_day', 'start_time_utc', 'end_day', 'end_time_utc')):
        start = datetime.fromisoformat(result['start_day'] + 'T' + result['start_time_utc'])
        end = datetime.fromisoformat(result['end_day'] + 'T' + result['end_time_utc'])
        if end < start:
            raise ValueError('End Day / End Time (UTC) is earlier than Start Day / Start Time (UTC)')
    return result


def metadata_attribute_value(value):
    """NetCDF attributes cannot store None, dicts, or arbitrary mixed lists."""
    if isinstance(value, np.generic):
        value = value.item()
    if value is None:
        return ''
    if isinstance(value, bool):
        return int(value)
    if isinstance(value, (dict, list, tuple, np.ndarray)):
        return json_text(value.tolist() if isinstance(value, np.ndarray) else value)
    if isinstance(value, (str, int, float)):
        return value
    raise TypeError(f'Unsupported metadata type: {type(value).__name__}')


def simplify_public_global_attributes(nc):
    """Clean public attributes and keep the tracer pair adjacent on disk."""
    code = public_product_code(nc)
    for name in REMOVED_PUBLIC_GLOBAL_ATTRIBUTES:
        if name in nc.ncattrs():
            nc.delncattr(name)
    nc.file_level_code = code
    tracer = {name: nc.getncattr(name) for name in
              ("selected_tracer_name", "selected_tracer_mz") if name in nc.ncattrs()}
    nc.sync()
    for name in tracer:
        nc.delncattr(name)
    # Flush deletions before re-adding, otherwise HDF5 may retain old disk positions.
    nc.sync()
    for name, value in tracer.items():
        nc.setncattr(name, value)


def write_global_metadata(nc, metadata):
    """Write every sample metadata field directly as a root/global attribute."""
    metadata = normalize_public_metadata(metadata)
    null_fields, json_fields = [], []
    for name, value in metadata.items():
        if name in REMOVED_PUBLIC_GLOBAL_ATTRIBUTES:
            continue
        if not re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*', name):
            raise ValueError(f'Invalid global attribute name: {name!r}')
        if value is None:
            null_fields.append(name)
        elif isinstance(value, (dict, list, tuple, np.ndarray)):
            json_fields.append(name)
        nc.setncattr(name, metadata_attribute_value(value))
    nc.metadata_null_fields = json.dumps(null_fields)
    nc.metadata_json_fields = json.dumps(json_fields)
    nc.min_altitude_units = 'm'
    nc.max_altitude_units = 'm'
    # The Excel Field Note column now owns this annotation.
    if 'field_note' in metadata and 'user_note' in nc.ncattrs():
        nc.delncattr('user_note')
    simplify_public_global_attributes(nc)


def update_completed_sample_metadata(state, sample_metadata, data_dir):
    """Refresh every completed public level after background counts become known."""
    sample_metadata = normalize_public_metadata(sample_metadata)
    save_json(Path(data_dir) / 'raw_metadata.json', sample_metadata)
    for level, path_string in state.get('netcdf_files', {}).items():
        if state.get('levels', {}).get(level, {}).get('status') != 'success':
            continue
        with Dataset(path_string, 'a') as nc:
            write_global_metadata(nc, sample_metadata)


FORMULA_TOKEN = re.compile(r"(?:\[(?:\d+)?([A-Z][a-z]?)\]|(?:\d+)?([A-Z][a-z]?))(\d*)")


def formula_element_counts(value):
    if pd.isna(value) or not str(value).strip():
        return {}
    formula, counts, position = str(value).strip(), {}, 0
    for match in FORMULA_TOKEN.finditer(formula):
        if formula[position:match.start()].strip():
            raise ValueError(f"Cannot parse CoreMS formula: {value!r}")
        element = match.group(1) or match.group(2)
        element = "H" if element in {"D", "T"} else element
        counts[element] = counts.get(element, 0) + int(match.group(3) or 1)
        position = match.end()
    if formula[position:].strip():
        raise ValueError(f"Cannot parse CoreMS formula: {value!r}")
    return {element: count for element, count in counts.items() if count > 0}


def molecular_class(counts):
    if not counts:
        return "unassigned"
    ordered = [e for e in ["C", "H", "N", "O", "S"] if e in counts]
    ordered += sorted(set(counts) - set(ordered))
    return "".join(ordered)


def validate_gap_settings(settings):
    fraction = settings.get("min_left_fraction", 0.8)
    if not np.isfinite(fraction) or not 0.5 < fraction <= 1.0:
        raise ValueError("gap min_left_fraction must be greater than 0.5 and at most 1")
    for name in ("min_formulas", "min_formulas_each_side"):
        if not isinstance(settings[name], (int, np.integer)) or settings[name] < 1:
            raise ValueError(f"gap {name} must be a positive integer")
    for name in ("min_width", "min_spacing_ratio"):
        if not np.isfinite(settings[name]) or settings[name] <= 0:
            raise ValueError(f"gap {name} must be finite and positive")
    if not isinstance(settings["round_decimals"], (int, np.integer)) or settings["round_decimals"] < 0:
        raise ValueError("gap round_decimals must be a nonnegative integer")


def detect_gap(formulas, settings):
    """Choose a supported upper-tail gap beyond the dominant formula distribution."""
    validate_gap_settings(settings)
    values = np.round(formulas["DBE_minus_O"].to_numpy(dtype=float), settings["round_decimals"])
    if not np.isfinite(values).all():
        raise ValueError("Nonfinite DBE-O values cannot define a gap")
    distinct, frequencies = np.unique(values, return_counts=True)
    gaps = np.diff(distinct)
    typical = float(np.median(gaps)) if len(gaps) else np.nan
    needed = max(settings["min_width"], settings["min_spacing_ratio"] * typical) if np.isfinite(typical) else np.inf
    fraction = float(settings.get("min_left_fraction", 0.8))
    # If several DBE-O values tie for the mode, protect all tied modes.
    mode_right = float(distinct[frequencies == frequencies.max()][-1]) if len(distinct) else np.nan
    rows = []
    for left, right in zip(distinct[:-1], distinct[1:]):
        nl, nr = int((values <= left).sum()), int((values >= right).sum())
        left_fraction = nl / len(values)
        support_ok = (len(values) >= settings["min_formulas"]
                      and right - left >= needed - 1e-10
                      and min(nl, nr) >= settings["min_formulas_each_side"])
        fraction_ok = left_fraction >= fraction - 1e-12
        mode_ok = left >= mode_right - 1e-10
        reasons = []
        if not support_ok:
            reasons.append("insufficient_gap_width_or_support")
        if not fraction_ok:
            reasons.append("too_few_formulas_on_retained_side")
        if not mode_ok:
            reasons.append("dominant_mode_on_removed_side")
        rows.append({"gap_left": left, "gap_right": right, "gap_width": right - left,
                     "n_left": nl, "n_right": nr, "left_fraction": left_fraction,
                     "support_qualified": support_ok, "retained_fraction_qualified": fraction_ok,
                     "right_of_mode": mode_ok, "qualifies": support_ok and fraction_ok and mode_ok,
                     "rejection_reason": "; ".join(reasons)})
    candidates = [row for row in rows if row["qualifies"]]
    # Only right-tail candidates reach this sort. Ties favor the higher cutoff.
    best = sorted(candidates, key=lambda row: (-row["gap_width"], -row["gap_left"]))[0] if candidates else None
    report = {"n_unique_formulas": len(values), "n_distinct_dbe_o": len(distinct),
              "typical_spacing": typical,
              "required_gap_width": needed if np.isfinite(needed) else np.nan,
              "gap_selection_version": "right_tail_body_guard_v1",
              "min_left_fraction": fraction, "dominant_mode_right": mode_right,
              "n_supported_gaps": int(sum(row["support_qualified"] for row in rows)),
              "n_body_safe_gaps": len(candidates),
              "gap_detected": best is not None,
              "gap_left": best["gap_left"] if best else np.nan,
              "gap_right": best["gap_right"] if best else np.nan,
              "selected_left_fraction": best["left_fraction"] if best else np.nan,
              "cutoff": (best["gap_left"] + best["gap_right"]) / 2 if best else np.nan}
    return report, rows


def internal_standard_formula_key(formula):
    """Compare chemical compositions, irrespective of element order and whitespace."""
    if pd.isna(formula) or not str(formula).strip():
        raise ValueError("An internal-standard comparison requires a molecular formula")
    counts = formula_element_counts(formula)
    if not counts:
        raise ValueError(f"Cannot parse molecular formula: {formula!r}")
    unsupported = set(counts) - {"C", "H", "O", "N", "S"}
    if unsupported:
        raise ValueError(f"Unsupported elements in molecular formula {formula!r}: {sorted(unsupported)}")
    return tuple(counts.get(element, 0) for element in ("C", "H", "O", "N", "S"))


def remove_internal_standard_peaks(rows, formulas, enabled):
    """Filter Level 3 unambiguous peaks by formula, preserving their intensities."""
    if not isinstance(enabled, bool):
        raise ValueError("remove_internal_standard_peaks must be a boolean setting")
    if not isinstance(formulas, (list, tuple)) or (enabled and not formulas):
        raise ValueError("Set a nonempty sequence of internal-standard formulas before enabling removal")
    target_keys = {}
    if enabled:
        for formula in formulas:
            key = internal_standard_formula_key(formula)
            if key in target_keys:
                raise ValueError(f"Duplicate internal-standard composition: {formula!r}")
            target_keys[key] = str(formula)
    if "Molecular Formula" not in rows.columns or "assign_type" not in rows.columns:
        raise ValueError("Expected unambiguous assignments with molecular formulas")
    if not rows["assign_type"].eq("unambiguous").all():
        raise ValueError("Internal-standard removal must receive only unambiguous peaks")
    remove = (rows["Molecular Formula"].map(
        lambda formula: internal_standard_formula_key(formula) in target_keys
    ) if enabled else pd.Series(False, index=rows.index))
    removed = rows.loc[remove].copy()
    kept = rows.loc[~remove].copy()
    report = {
        "enabled": enabled,
        "matching_rule": "C/H/O/N/S atom counts, independent of whitespace or element order",
        "configured_formulas": list(formulas),
        "n_removed_peaks": int(removed["m/z"].nunique()),
        "removed_formulas": sorted(removed["Molecular Formula"].astype(str).unique().tolist()),
    }
    return kept, removed, report


def clean_dbe_o_assignments(all_rows, settings):
    audit = all_rows.copy().reset_index(drop=True)
    counts = audit["Molecular Formula"].map(formula_element_counts)
    audit["chemical_class"] = counts.map(molecular_class)
    audit["O_count"] = counts.map(lambda item: item.get("O", 0))
    audit["DBE_minus_O"] = pd.to_numeric(audit["DBE"], errors="coerce") - audit["O_count"]
    assigned = audit["chemical_class"].ne("unassigned")
    if not np.isfinite(audit.loc[assigned, "DBE_minus_O"].to_numpy(dtype=float)).all():
        raise ValueError("Missing/nonfinite CoreMS DBE on assigned formulas; cannot run automatic Level 3 cleaning")
    audit["dbe_o_keep"] = True
    audit["dbe_o_cutoff"] = np.nan
    audit["dbe_o_rule"] = "unassigned"
    reports, gap_rows = [], []
    for label in sorted(audit.loc[assigned, "chemical_class"].unique()):
        subset = audit.loc[audit["chemical_class"].eq(label)].copy()
        subset["_formula_key"] = subset["Molecular Formula"].map(topn.normalize_formula)
        inconsistency = subset.groupby("_formula_key")["DBE_minus_O"].nunique().gt(1)
        if inconsistency.any():
            raise ValueError("The same molecular formula has conflicting DBE-O values")
        unique = subset.drop_duplicates("_formula_key")
        report, intervals = detect_gap(unique, settings)
        if report["gap_detected"]:
            cutoff, rule = report["cutoff"], "automatic_gap"
        elif label in {"CHOS", "CHNOS"}:
            cutoff, rule = 1.0, "fallback_CHOS_CHNOS_gt_1"
        else:
            cutoff, rule = np.nan, "no_reliable_gap_no_filter"
        mask = audit["chemical_class"].eq(label)
        audit.loc[mask, "dbe_o_cutoff"] = cutoff
        audit.loc[mask, "dbe_o_rule"] = rule
        if np.isfinite(cutoff):
            audit.loc[mask, "dbe_o_keep"] = audit.loc[mask, "DBE_minus_O"].le(cutoff)
        report.update(chemical_class=label, applied_cutoff=cutoff, rule=rule,
                      n_removed_rows=int((mask & ~audit["dbe_o_keep"]).sum()))
        reports.append(report)
        gap_rows.extend({"chemical_class": label, **row} for row in intervals)
    cleaned = audit.loc[audit["dbe_o_keep"]].copy()
    surviving = cleaned.loc[cleaned["assign_type"].ne("unassigned")].groupby("m/z")["Molecular Formula"].nunique()
    cleaned["n_formulas_after_dbe_o"] = cleaned["m/z"].map(surviving).fillna(0).astype(int)
    tables = {"assignments_before_dbe_o": all_rows.copy(), "dbe_o_audit": audit,
              "dbe_o_removed": audit.loc[~audit["dbe_o_keep"]].copy(), "assignments_all": cleaned,
              "gap_summary": pd.DataFrame(reports), "gap_intervals": pd.DataFrame(gap_rows)}
    tables.update({label: cleaned.loc[cleaned["assign_type"].eq(label)].copy()
                   for label in ["unambiguous", "ambiguous", "unassigned"]})
    return tables


def snapshot_average(spectrum):
    """Copy Level 1 arrays before any in-place Level 2 modifications."""
    profile_mz = np.array(spectrum.mz_exp_profile, dtype=float, copy=True)
    profile_intensity = np.array(spectrum.abundance_profile, dtype=float, copy=True)
    if not len(profile_mz) or len(profile_mz) != len(profile_intensity):
        raise ValueError("Expected a nonempty averaged profile with matching m/z and intensity arrays")
    if not np.isfinite(profile_mz).all() or not np.isfinite(profile_intensity).all() or (np.diff(profile_mz) < 0).any():
        raise ValueError("Invalid averaged profile arrays")
    peaks = pd.DataFrame([{"mz": float(p.mz_exp), "intensity": float(p.abundance),
                           "resolving_power": positive_or_nan(p.resolving_power),
                           "snr": positive_or_nan(p.signal_to_noise)} for p in spectrum.mspeaks],
                         columns=["mz", "intensity", "resolving_power", "snr"])
    if not np.isfinite(peaks[["mz", "intensity"]].to_numpy()).all():
        raise ValueError("Nonfinite picked m/z/intensity in the averaged spectrum")
    return {"profile_mz": profile_mz, "profile_intensity": profile_intensity, "peaks": peaks}


def assignment_tables(spectrum, classified):
    all_rows = classified["all"].copy().reset_index(drop=True)
    all_rows["assignment_row"] = np.arange(len(all_rows), dtype=np.int64)
    tables = {"assignments_all": all_rows}
    tables.update({label: all_rows.loc[all_rows["assign_type"].eq(label)].copy()
                   for label in ["unambiguous", "ambiguous", "unassigned"]})
    tables["averaged_profile"] = pd.DataFrame({"mz": np.array(spectrum.mz_exp_profile, copy=True),
                                               "abundance": np.array(spectrum.abundance_profile, copy=True)})
    return tables


def assign_existing_average(spectrum, scans, cfg):
    """Apply the original module's post-average operations to the same Level 1 spectrum."""
    corems = topn._import_corems()
    meta = {"n_scans_averaged": len(scans), "selected_scans": list(scans), "n_peaks_initial": len(spectrum)}
    if cfg.rp_filter_enabled:
        meta["rp_filter"] = topn._orbitrap_rp_filter(spectrum, float(cfg.rp_mz_ref), float(cfg.rp_fraction), float(cfg.rp_mz_window))
    meta["n_peaks_after_rp"] = len(spectrum)
    try:
        spectrum.run_noise_threshold_calc()
        meta["noise_threshold"] = float(spectrum.get_noise_threshold()[1][0])
    except Exception as exc:
        meta["noise_threshold_error"] = repr(exc)
    if cfg.recalibration_enabled:
        corems["MzDomainCalibration"](spectrum, str(cfg.reference_mass_list_path)).run()
        meta["recalibration"] = {"reference_mass_list": str(cfg.reference_mass_list_path), **topn._calibration_metrics(spectrum)}
    topn._configure_molecular_search(spectrum, cfg)
    corems["SearchMolecularFormulas"](spectrum, first_hit=bool(cfg.first_hit)).run_worker_mass_spectrum()
    optional(lambda: spectrum.percentile_assigned())
    return assignment_tables(spectrum, topn.classify_assignments(spectrum.to_dataframe())), meta


def export_tables(group, tables):
    """Write processed tables to NetCDF groups without CSV sidecars."""
    for name, frame in tables.items():
        write_nc_table(group, name, frame)


def all_candidate_regions(region_tables, tracer_masses):
    tables = [table.assign(tracer_name=name, tracer_mz=tracer_masses[name])
              for name, table in region_tables.items()]
    if tables:
        nonempty = [table for table in tables if not table.empty]
        return pd.concat(nonempty, ignore_index=True) if nonempty else tables[0].iloc[:0].copy()
    return pd.DataFrame(columns=["tracer_name", "tracer_mz"])


def write_selection_diagnostics(path, candidates, regions, selected, decision):
    """Persist exact scan-selection audits before Level 1, including failed tracer runs."""
    with Dataset(path, "w", format="NETCDF4") as nc:
        nc.schema_version = "corems-three-levels-selection-1.0"
        nc.processing_status = decision["status"]
        nc_text(nc, "sample_scan_selection_json", json_text(decision))
        export_tables(nc, {"sample_scan_audit": candidates,
                           "all_tracer_candidate_regions": regions,
                           "selected_sample_scans": (selected if selected is not None
                                                     else candidates.iloc[:0].copy())})


def append_background_selection_diagnostics(path, candidates, selected, mode):
    with Dataset(path, "a") as nc:
        nc_text(nc, "background_mode", mode)
        export_tables(nc, {"background_candidates": candidates,
                           "selected_background_scans": selected})


def write_level1(nc, snapshot, scans):
    return write_public_level1(nc, snapshot, scans)


def validate_level_netcdf(path, level):
    result = validate_public_level(path, level)
    if level == "level3":
        with Dataset(path) as nc:
            if "van_krevelen_png" not in nc.variables:
                raise AssertionError("Level 3 is missing its van Krevelen image")
            png = np.asarray(nc.variables["van_krevelen_png"][:], dtype=np.uint8).tobytes()
            if not png.startswith(b"\x89PNG\r\n\x1a\n"):
                raise AssertionError("Level 3 contains an invalid PNG image")
    return result


def write_diagnostic_level(path, level, writer):
    """Keep full CoreMS tables and cleaning audits outside the public files."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    partial = path.with_suffix(".partial.nc")
    try:
        with Dataset(partial, "w", format="NETCDF4") as nc:
            nc.schema_version = "corems-three-levels-diagnostics-1.0"
            nc.output_level = level
            writer(nc)
        partial.replace(path)
    except BaseException:
        partial.unlink(missing_ok=True)
        raise


def render_level3_van_krevelen(nc, frame, save_path):
    """Save the final chemical-class plot as a PNG file and embedded PNG bytes."""
    from io import BytesIO

    counts = frame["Molecular Formula"].map(formula_element_counts)
    carbon = counts.map(lambda item: item.get("C", 0)).to_numpy(dtype=float)
    oxygen = counts.map(lambda item: item.get("O", 0)).to_numpy(dtype=float)
    hydrogen = counts.map(lambda item: item.get("H", 0)).to_numpy(dtype=float)
    classes = counts.map(molecular_class)
    fig, ax = plt.subplots(figsize=(7, 5))
    palette = {"CHO": "tab:green", "CHOS": "tab:red", "CHNO": "tab:blue", "CHNOS": "tab:pink"}
    for label in classes.unique():
        valid = (classes == label).to_numpy() & (carbon > 0)
        ax.scatter(oxygen[valid] / carbon[valid], hydrogen[valid] / carbon[valid],
                   s=14, alpha=0.65, color=palette.get(label, "0.5"), label=label)
    ax.set(xlabel="O/C", ylabel="H/C", title="c1: retained assignments")
    if len(frame):
        ax.legend()
    fig.tight_layout()
    stream = BytesIO()
    try:
        fig.savefig(stream, format="png", dpi=180)
    finally:
        plt.close(fig)
    payload = stream.getvalue()
    Path(save_path).write_bytes(payload)
    nc.createDimension("van_krevelen_png_byte", len(payload))
    image = nc.createVariable("van_krevelen_png", "u1", ("van_krevelen_png_byte",),
                              zlib=True, complevel=NETCDF_COMPRESSION_LEVEL)
    image[:] = np.frombuffer(payload, dtype=np.uint8)
    image.mime_type = "image/png"
    image.source = "Final c1 retained formula assignments"
    image.description = "PNG image of O/C versus H/C, colored by molecular class"


def write_level_file(path, level, manifest, writer, sample_metadata, background_metadata=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    partial = path.with_suffix(".partial.nc")
    try:
        with Dataset(partial, "w", format="NETCDF4") as nc:
            nc.file_level_code = {"level1":"a1", "level2":"b1", "level3":"c1"}[level]
            nc.processing_status = "running"
            nc.raw_metadata_file = "raw_metadata.json"
            nc.sample_raw_filename = Path(manifest["sample_raw_file"]).name
            nc.n_sample_scans_averaged = len(manifest["selected_sample_scans"])
            nc.selected_tracer_name = str(manifest.get("selected_sample_tracer_name") or "")
            if level == "level3":
                nc.background_mode = manifest["level3_options"]["background_mode"]
                nc.background_removal_rule = manifest["level3_options"]["background_removal_mode"]
                nc.sample_background_ratio_cutoff = float(manifest["level3_options"]["sample_background_ratio_cutoff"])
                nc.n_background_scans_averaged = len(manifest["selected_background_scans"])
            details = writer(nc)
            write_global_metadata(nc, sample_metadata)
            nc.output_filename = path.name
            nc.file_level_code = {"level1":"a1", "level2":"b1", "level3":"c1"}[level]
        validation = validate_level_netcdf(partial, level)
        with Dataset(partial, "a") as nc:
            nc.processing_status = "success"
        partial.replace(path)
        print(f"Completed {dict(level1='a1', level2='b1', level3='c1')[level]}: {path}")
        return {"status": "success", "netcdf": str(path), **details, **validation}
    except BaseException as exc:
        if partial.exists():
            try:
                with Dataset(partial, "a") as nc:
                    nc.processing_status = "failed"
                    nc.error = f"{type(exc).__name__}: {exc}"
            except Exception:
                pass
            failed_path = path.parent.parent / "diagnostics" / f"{level}.failed.partial.nc"
            partial.replace(failed_path)
        raise


def save_selection_diagnostics(frame, selected, background, mode, analysis, folder,
                               tracer_name, tracer_mz):
    """Plot one original tracer EIC, its intervals/cutoff, and any selected scan IDs."""
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
    scans = frame["scan"].to_numpy(dtype=int)
    raw = pd.to_numeric(frame[tracer_name], errors="coerce").to_numpy(dtype=float)
    axes[0].plot(scans, frame["tic"], lw=0.8, color="0.45")
    axes[0].set(ylabel="TIC", title=f"MS1: {tracer_mz:.5f} ({tracer_name}); {analysis['status']}")
    axes[1].plot(scans, raw, lw=0.8, color="tab:blue", label=f"{tracer_mz:.5f} raw EIC")
    if np.isfinite(analysis["cutoff"]):
        axes[1].axhline(analysis["cutoff"], ls="--", color="tab:orange", label="Positive-only Otsu cutoff")
    if selected is not None and not selected.empty:
        axes[1].scatter(selected["scan"], selected[tracer_name], s=20, color="tab:red", label="Selected sample", zorder=3)
    if mode == "same_file" and background is not None and not background.empty:
        axes[1].scatter(background["scan"], background[tracer_name], s=24, marker="x",
                        color="black", label="Background", zorder=3)
    axes[1].set(ylabel="Raw EIC intensity")
    axes[1].legend(fontsize=8)
    axes[1].set_xlabel("Thermo scan number")
    for i, (start, stop) in enumerate(analysis["regions"]):
        left, right = scans[start], scans[stop - 1]
        for ax in axes:
            ax.axvspan(left, right, color="tab:green", alpha=0.13,
                       label="Accepted tracer ROI" if ax is axes[1] and i == 0 else None)
    if analysis["regions"]:
        axes[1].legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(folder / f"01_{int(round(tracer_mz))}_roi_selection.png", dpi=180)
    plt.close(fig)


def save_unrestricted_sample_diagnostics(frame, selected, mode, folder):
    """Show the original TIC and exactly which scan IDs were drawn for the sample average."""
    fig, ax = plt.subplots(figsize=(11, 3))
    ax.plot(frame["scan"], frame["tic"], lw=0.8, color="0.5", label="Eligible MS1 TIC")
    ax.scatter(selected["scan"], selected["tic"], s=19, color="tab:red", label="Selected sample")
    ax.set(xlabel="Thermo scan number", ylabel="TIC", title=f"Sample selection: {mode}")
    ax.legend()
    fig.tight_layout()
    fig.savefig(Path(folder) / "01_sample_selection_tic.png", dpi=180)
    plt.close(fig)


def save_gap_diagnostics(tables, folder, prefix):
    audit, summary = tables["dbe_o_audit"], tables["gap_summary"]
    if summary.empty:
        return
    labels = summary["chemical_class"].tolist()
    fig, axes = plt.subplots(1, len(labels), figsize=(max(5, 3.8 * len(labels)), 3.8), squeeze=False)
    for ax, label in zip(axes.ravel(), labels):
        rows = audit.loc[audit["chemical_class"].eq(label)].copy()
        rows["_formula_key"] = rows["Molecular Formula"].map(topn.normalize_formula)
        rows = rows.drop_duplicates("_formula_key")
        x = rows["DBE_minus_O"].to_numpy(dtype=float)
        lo, hi = np.floor(x.min()), np.ceil(x.max())
        bins = np.arange(lo - 0.25, hi + 0.76, 0.5)
        ax.hist(x, bins=bins, color="0.8", label="Before")
        ax.hist(rows.loc[rows["dbe_o_keep"], "DBE_minus_O"], bins=bins, color="tab:green", alpha=0.7, label="Retained")
        decision = summary.loc[summary["chemical_class"].eq(label)].iloc[0]
        if np.isfinite(decision["applied_cutoff"]):
            ax.axvline(decision["applied_cutoff"], color="tab:red", ls="--", label=f"Cutoff {decision['applied_cutoff']:g}")
        if decision["gap_detected"]:
            ax.axvspan(decision["gap_left"], decision["gap_right"], color="tab:orange", alpha=0.15)
        ax.set(title=f"{label}: {int((~rows['dbe_o_keep']).sum())} removed", xlabel="DBE - O", ylabel="Unique formulas")
        ax.legend(fontsize=8)
    fig.suptitle(f"{prefix}: automatic DBE-O cleaning (trial settings)")
    fig.tight_layout()
    fig.savefig(Path(folder) / f"{prefix}_dbe_o_cleaning.png", dpi=180)
    plt.close(fig)


def validate_settings(options):
    validate_gap_settings(options["gap_settings"])
    if not isinstance(options["top_n_sample"], int) or options["top_n_sample"] < 1:
        raise ValueError("top_n_sample must be a positive integer")
    if options["background_mode"] not in {"same_file", "external_file", "none"}:
        raise ValueError("BACKGROUND_MODE must be same_file, external_file, or none")
    if not isinstance(options["sample_random_seed"], int) or options["sample_random_seed"] < 0:
        raise ValueError("sample_random_seed must be a nonnegative integer")
    unknown = set(options["metadata_fields"]) - set(RAW_ACQUISITION_METADATA_FIELDS) - set(PROCESSING_METADATA_FIELDS)
    if unknown:
        raise ValueError(f"Unknown metadata fields: {sorted(unknown)}")
    if options["eic_tolerance_ppm"] <= 0:
        raise ValueError("EIC tolerance must be positive")
    if options["background_mode"] != "same_file":
        return
    if not candidate_rows(options["modules"]) or len({name for _, name, _ in candidate_rows(options["modules"])}) != len(candidate_rows(options["modules"])):
        raise ValueError("Candidate ion names must be unique across nonempty modules")
    if not all(np.isfinite(mz) and mz > 0 for _, _, mz in candidate_rows(options["modules"])):
        raise ValueError("Tracer masses must be positive and finite")
    priority = options["tracer_priority"]
    if not priority or len(priority) != len(set(priority)):
        raise ValueError("Tracer priority must be a nonempty sequence of unique ion names")
    if not set(priority).issubset({name for _, name, _ in candidate_rows(options["modules"])}):
        raise ValueError("Every priority tracer must be among the extracted EICs")
    roi = options["roi_settings"]
    if (roi["max_interior_gap_scans"] < 0 or roi["min_region_scans"] < 1
            or not 0 < roi["min_high_fraction"] <= 1
            or roi["min_inside_to_low_fold"] < 1
            or roi["min_low_reference_scans"] < 1):
        raise ValueError("Invalid sample-region criteria")


def validate_background_settings(options):
    if options["background_mode"] not in {"same_file", "external_file", "none"}:
        raise ValueError("Unknown background mode")
    if not isinstance(options["top_n_background"], int) or options["top_n_background"] < 1:
        raise ValueError("top_n_background must be a positive integer")
    gap = options["gap_settings"]
    if gap["min_formulas"] < 2 or gap["min_formulas_each_side"] < 1 or gap["min_width"] <= 0 or gap["min_spacing_ratio"] <= 1:
        raise ValueError("Invalid gap criteria")
    enabled = options["remove_internal_standard_peaks"]
    formulas = options["internal_standard_formulas"]
    if not isinstance(enabled, bool):
        raise ValueError("REMOVE_INTERNAL_STANDARD_PEAKS must be True or False")
    if enabled:
        if not isinstance(formulas, (list, tuple)) or not formulas:
            raise ValueError("List the actual internal-standard formulas before enabling removal")
        keys = [internal_standard_formula_key(formula) for formula in formulas]
        if len(keys) != len(set(keys)):
            raise ValueError("INTERNAL_STANDARD_FORMULAS contains duplicate compositions")


def run_three_level_workflow(raw_path, output_root, cfg, options):
    validate_settings(options)
    field_metadata = normalized_field_metadata(options.get("field_metadata"))
    new_file_name = normalize_output_stem(options.get("new_file_name"))
    raw_path = Path(raw_path).resolve()
    if not raw_path.is_file():
        raise FileNotFoundError(raw_path)
    if cfg.recalibration_enabled and not Path(cfg.reference_mass_list_path).is_file():
        raise FileNotFoundError(cfg.reference_mass_list_path)
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
    run_dir = Path(output_root) / topn.safe_output_key(raw_path.stem) / stamp
    run_dir.mkdir(parents=True, exist_ok=False)
    data_dir = run_dir / "data"
    data_dir.mkdir()
    diagnostics_dir = run_dir / "diagnostics"
    diagnostics_dir.mkdir()
    plots = diagnostics_dir / "plots"
    plots.mkdir()
    state = {"status": "running", "run_directory": str(run_dir), "netcdf_files": {},
             "diagnostic_files": {},
             "levels": {level: {"status": "pending"} for level in ["level1", "level2", "level3"]}}
    state_path = diagnostics_dir / "three_level_status.json"
    save_json(state_path, state)
    sample_options = {name: options[name] for name in
                      ["modules", "tracer_priority", "top_n_sample", "sample_random_seed", "ms_filter",
                       "eic_tolerance_ppm", "roi_settings", "metadata_fields"]}
    manifest = {"schema_version": PUBLIC_SCHEMA_VERSION, "revision": 12, "run_id": stamp,
                "sample_raw_file": str(raw_path), "corems_config": asdict(cfg),
                "sample_selection_options": sample_options,
                "processing_module_sha256": hashlib.sha256(Path(topn.__file__).read_bytes()).hexdigest()}
    manifest["field_metadata"] = field_metadata
    manifest["new_file_name"] = new_file_name
    save_json(diagnostics_dir / "three_level_config.json", manifest)
    active_level = None
    try:
        mode = options["background_mode"]
        tracer_masses = ({name: mz for _, name, mz in candidate_rows(options["modules"])}
                         if mode == "same_file" else {})
        if mode == "same_file":
            print("Extracting unsmoothed MS1 EICs (priority: " +
                  ", ".join(f"{tracer_masses[name]:.5f}" for name in options["tracer_priority"]) + ")...")
        else:
            print(f"Reading MS1 TIC for sample selection (background mode: {mode})...")
        signals = extract_scan_signals(raw_path, options["modules"] if mode == "same_file" else {},
                                       options["ms_filter"], options["eic_tolerance_ppm"])
        if signals.empty:
            raise ValueError("No MS1 scans returned from the sample RAW")
        eligible = np.isfinite(pd.to_numeric(signals["tic"], errors="coerce")) & signals["tic"].gt(0)
        candidates = signals.loc[eligible].sort_values("scan").reset_index(drop=True).copy()
        if candidates.empty:
            raise ValueError("No positive-TIC MS1 scans were returned from the sample RAW")
        selection_dir = diagnostics_dir / "selection"
        selection_dir.mkdir()
        selection_netcdf = selection_dir / "selection_diagnostics.nc"
        state["diagnostic_files"]["selection"] = str(selection_netcdf)
        if mode == "same_file":
            analyses, summaries, region_tables = {}, {}, {}
            for name in options["tracer_priority"]:
                mz = tracer_masses[name]
                prefix = str(int(round(mz)))
                analysis_for_ion = analyze_eic(candidates[name], options["roi_settings"])
                analyses[name] = analysis_for_ion
                candidates[f"{prefix}_raw_above_cutoff"] = analysis_for_ion["high_mask"]
                candidates[f"{prefix}_accepted_roi"] = analysis_for_ion["mask"]
                candidates[f"{prefix}_otsu_cutoff"] = analysis_for_ion["cutoff"]
                candidates[f"{prefix}_selected_sample"] = False
                region_rows = [{**row, "first_scan": int(candidates.iloc[row["first_index"]]["scan"]),
                                "last_scan": int(candidates.iloc[row["stop_index"] - 1]["scan"])}
                               for row in analysis_for_ion["region_details"]]
                region_table_for_ion = pd.DataFrame(region_rows, columns=[
                    "first_index", "stop_index", "n_scans", "raw_high_fraction",
                    "n_low_scans_before", "n_low_scans_after", "touches_file_start",
                    "touches_file_end", "median_intensity",
                    "low_reference_median", "inside_to_low_fold", "contrast_check",
                    "passed", "first_scan", "last_scan"])
                region_tables[name] = region_table_for_ion
                cutoff = analysis_for_ion["cutoff"]
                summary = {"tracer_name": name, "tracer_mz": mz,
                           "status": analysis_for_ion["status"], "reason": analysis_for_ion["reason"],
                           "cutoff": float(cutoff) if np.isfinite(cutoff) else None,
                           "regions": [{"first_scan": int(candidates.iloc[row["first_index"]]["scan"]),
                                        "last_scan": int(candidates.iloc[row["stop_index"] - 1]["scan"]),
                                        "n_scans": int(row["n_scans"]),
                                        "n_low_scans_before": int(row["n_low_scans_before"]),
                                        "n_low_scans_after": int(row["n_low_scans_after"]),
                                        "touches_file_start": bool(row["touches_file_start"]),
                                        "touches_file_end": bool(row["touches_file_end"])}
                                       for row in analysis_for_ion["region_details"] if row["passed"]]}
                summaries[name] = summary
                save_json(selection_dir / f"{prefix}_selection_decision.json", summary)
                save_selection_diagnostics(candidates, None, None, None, analysis_for_ion,
                                           plots, tracer_name=name, tracer_mz=mz)

            attempted = []
            tracer_name = None
            for name in options["tracer_priority"]:
                attempted.append(summaries[name])
                if analyses[name]["status"] == "candidate":
                    tracer_name = name
                    break
            tracer_mz = tracer_masses[tracer_name] if tracer_name else None
            analysis = analyses[tracer_name] if tracer_name else None
            n_roi_scans = int(analysis["mask"].sum()) if analysis is not None else 0
            rejected = [f"{row['tracer_mz']:.5f} {row['tracer_name']}: {row['reason']}"
                        for row in attempted if row["status"] != "candidate"]
            fallback_reason = "; ".join(rejected) if rejected else "first_priority_tracer_accepted"
            decision = {"status": "selected" if tracer_name else "failed",
                        "background_mode": mode,
                        "priority": list(options["tracer_priority"]),
                        "selected_tracer_name": tracer_name, "selected_tracer_mz": tracer_mz,
                        "n_accepted_roi_scans": n_roi_scans,
                        "n_eligible_ms1_scans": len(candidates),
                        "n_selected_sample_scans": min(n_roi_scans, options["top_n_sample"]),
                        "fallback_reason": fallback_reason, "attempted_tracers": attempted,
                        "random_seed": None,
                        "selection_rule": ("no_accepted_tracer" if tracer_name is None else
                                           "all_scans_in_accepted_roi_at_or_below_limit"
                                           if n_roi_scans <= options["top_n_sample"] else
                                           "top_raw_chosen_tracer_eic_within_accepted_roi")}
            save_json(selection_dir / "sample_scan_selection_decision.json", decision)
            candidates["selected_sample_scan"] = False
            if tracer_name is None:
                write_selection_diagnostics(selection_netcdf, candidates,
                                            all_candidate_regions(region_tables, tracer_masses),
                                            None, decision)
                raise ValueError("No applicable sample region for any priority tracer: " + "; ".join(rejected))

            selected = select_sample_scans(candidates, options["top_n_sample"], analysis, tracer_name)
            sample_scans = selected["scan"].astype(int).tolist()
            candidates["selected_sample_scan"] = candidates["scan"].isin(sample_scans)
            if tracer_name is not None:
                candidates[f"{int(round(tracer_mz))}_selected_sample"] = candidates["selected_sample_scan"]
            write_selection_diagnostics(selection_netcdf, candidates,
                                        all_candidate_regions(region_tables, tracer_masses),
                                        selected, decision)
            save_selection_diagnostics(candidates, selected, None, None, analysis,
                                       plots, tracer_name=tracer_name, tracer_mz=tracer_mz)
            print(f"Selected {len(sample_scans)} scans from {n_roi_scans} accepted ROI scans "
                  f"using {tracer_name} (m/z {tracer_mz:.5f}).")
        else:
            tracer_name, tracer_mz, analysis, n_roi_scans = None, None, None, None
            attempted, fallback_reason, summaries, region_tables = [], None, {}, {}
            selected = select_unrestricted_sample_scans(candidates, options["top_n_sample"],
                                                        options["sample_random_seed"])
            sample_scans = selected["scan"].astype(int).tolist()
            candidates["selected_sample_scan"] = candidates["scan"].isin(sample_scans)
            decision = {"status": "selected", "background_mode": mode,
                        "priority": [], "selected_tracer_name": None, "selected_tracer_mz": None,
                        "n_accepted_roi_scans": None, "n_eligible_ms1_scans": len(candidates),
                        "n_selected_sample_scans": len(sample_scans),
                        "fallback_reason": None, "attempted_tracers": [],
                        "selection_rule": selected["selection_rule"].iloc[0],
                        "random_seed": (options["sample_random_seed"] if len(candidates) > options["top_n_sample"]
                                        else None)}
            save_json(selection_dir / "sample_scan_selection_decision.json", decision)
            write_selection_diagnostics(selection_netcdf, candidates,
                                        all_candidate_regions(region_tables, tracer_masses),
                                        selected, decision)
            save_unrestricted_sample_diagnostics(candidates, selected, mode, plots)
            print(f"Selected {len(sample_scans)} of {len(candidates)} eligible MS1 scans: "
                  f"{decision['selection_rule']}.")
        acquisition_fields = [name for name in options["metadata_fields"]
                              if name in RAW_ACQUISITION_METADATA_FIELDS]
        sample_metadata = read_raw_metadata(raw_path, acquisition_fields)
        derived_metadata = {"selected_tracer_name": tracer_name, "selected_tracer_mz": tracer_mz,
                            "n_sample_scans_averaged": len(sample_scans),
                            "n_background_scans_averaged": 0 if mode == "none" else None}
        sample_metadata.update({name: derived_metadata[name] for name in options["metadata_fields"]
                                if name in derived_metadata})
        sample_metadata = {name: sample_metadata[name] for name in options["metadata_fields"]}
        sample_metadata.update(field_metadata)
        sample_metadata = normalize_public_metadata(sample_metadata)
        background_metadata = None
        save_json(data_dir / "raw_metadata.json", sample_metadata)
        manifest.update(selected_sample_scans=sample_scans,
                        sample_selection_rule=selected["selection_rule"].iloc[0],
                        selected_sample_tracer_name=tracer_name, selected_sample_tracer_mz=tracer_mz,
                        sample_tracer_fallback_reason=fallback_reason,
                        sample_tracer_attempts=attempted, sample_scan_selection=decision,
                        n_ms1_scans_returned=len(signals), n_eligible_positive_tic_scans=len(candidates),
                        n_accepted_roi_scans=n_roi_scans,
                        sample_roi_status=analysis["status"] if analysis is not None else "not_applicable",
                        sample_roi_reason=analysis["reason"] if analysis is not None else None,
                        sample_roi_cutoff=summaries[tracer_name]["cutoff"] if tracer_name else None,
                        sample_roi_regions=summaries[tracer_name]["regions"] if tracer_name else [])
        save_json(diagnostics_dir / "three_level_config.json", manifest)
        state.update(n_sample_scans=len(sample_scans))

        def complete(level, writer):
            state["levels"][level] = {"status": "running"}
            save_json(state_path, state)
            path = data_dir / public_level_filename(new_file_name, level)
            state["levels"][level] = write_level_file(path, level, manifest, writer, sample_metadata, background_metadata)
            state["netcdf_files"][level] = str(path)
            save_json(state_path, state)

        active_level = "level1"
        print("Averaging selected sample scans and picking peaks...")
        spectrum = topn.average_profile_spectrum(raw_path, sample_scans, cfg)
        snapshot = snapshot_average(spectrum)
        complete("level1", lambda nc: write_level1(nc, snapshot, sample_scans))

        active_level = "level2"
        sample_result = {}

        def level2_writer(nc):
            tables, processing = assign_existing_average(spectrum, sample_scans, cfg)
            sample_result.update(tables=tables, processing=processing)
            selection_tables = ({"scan_eic_roi_audit": candidates,
                                 "sample_candidate_regions": region_tables[tracer_name],
                                 "all_tracer_candidate_regions": all_candidate_regions(region_tables, tracer_masses)}
                                if mode == "same_file" else {"scan_selection_audit": candidates})
            details = write_public_level2(nc, tables, formula_element_counts)
            nc.background_removal_performed = 0

            diagnostic_path = diagnostics_dir / "level2_audit.nc"
            def write_level2_audit(audit_nc):
                audit_group = audit_nc.createGroup("level2")
                export_tables(audit_group, {**selection_tables,
                                            "selected_sample_scans": selected, **tables})
                nc_text(audit_group, "processing_metadata_json", json_text(processing))
            write_diagnostic_level(diagnostic_path, "level2", write_level2_audit)
            state["diagnostic_files"]["level2"] = str(diagnostic_path)
            return {**details, "background_removal_performed": False}


        complete("level2", level2_writer)
        del spectrum, snapshot

        # All background work, including configuration and RAW path validation, starts here.
        active_level = "level3"
        manifest["level3_options"] = {name: options[name] for name in
                                       ["background_mode", "background_raw_path", "top_n_background",
                                        "background_seed", "background_removal_mode",
                                        "sample_background_ratio_cutoff", "gap_settings",
                                        "remove_internal_standard_peaks", "internal_standard_formulas"]}
        save_json(diagnostics_dir / "three_level_config.json", manifest)
        validate_background_settings(options)
        bg_path = raw_path
        if mode == "external_file":
            if options["background_raw_path"] is None:
                raise ValueError("Set BACKGROUND_RAW_PATH for external_file mode")
            bg_path = Path(options["background_raw_path"]).resolve()
            if not bg_path.is_file():
                raise FileNotFoundError(bg_path)
            if bg_path == raw_path:
                raise ValueError("The external background must be a different RAW file")

        if mode == "none":
            background_candidates = candidates.iloc[:0].copy()
        elif mode == "external_file":
            background_candidates = extract_scan_signals(bg_path, {}, options["ms_filter"],
                                                         options["eic_tolerance_ppm"])
        else:
            background_candidates = candidates
        background = select_background_scans(background_candidates, sample_scans, mode,
                                              options["top_n_background"], options["background_seed"],
                                              roi_mask=analysis["mask"] if mode == "same_file" else None,
                                              tracer_name=tracer_name)
        bg_scans = background["scan"].astype(int).tolist()
        if mode == "same_file" and set(sample_scans) & set(bg_scans):
            raise AssertionError("Same-file sample and background scans overlap")
        append_background_selection_diagnostics(selection_netcdf, background_candidates, background, mode)
        if mode == "same_file":
            save_selection_diagnostics(candidates, selected, background, mode, analysis,
                                       plots, tracer_name=tracer_name, tracer_mz=tracer_mz)
        if mode == "external_file" and not background_candidates.empty:
            fig, ax = plt.subplots(figsize=(11, 3))
            ax.plot(background_candidates["scan"], background_candidates["tic"], color="0.5", label="External RAW")
            if not background.empty:
                ax.scatter(background["scan"], background["tic"], s=22, color="tab:red", label="Random background scans")
            ax.set(xlabel="Thermo scan number", ylabel="TIC", title="External background selection")
            ax.legend()
            fig.tight_layout()
            fig.savefig(plots / "02_external_background_selection.png", dpi=180)
            plt.close(fig)
        if mode == "external_file" and bg_scans:
            background_metadata = normalize_public_metadata(read_raw_metadata(bg_path, acquisition_fields))
            save_json(data_dir / "background_raw_metadata.json", background_metadata)
        manifest.update(background_raw_file=str(bg_path) if mode != "none" else None,
                        selected_background_scans=bg_scans)
        save_json(diagnostics_dir / "three_level_config.json", manifest)
        state.update(n_background_scans=len(bg_scans))
        if mode == "none" or not bg_scans:
            reason = "background_mode_none" if mode == "none" else "no_eligible_background_scans"
            state["levels"]["level3"] = {"status": "skipped", "reason": reason,
                                            "chemical_cleaning_applied": False,
                                            "background_removal_performed": False}
            print(f"Level 3 skipped: {reason}.")
        elif len(bg_scans) < options["top_n_background"]:
            print(f"Using all {len(bg_scans)} eligible background scans (fewer than the requested maximum).")
        save_json(state_path, state)

        if mode != "none" and bg_scans:
            if "n_background_scans_averaged" in sample_metadata:
                sample_metadata["n_background_scans_averaged"] = len(bg_scans)
            def level3_writer(nc):
                sample_unambiguous = sample_result["tables"]["unambiguous"]
                bg_spectrum, bg_classified, bg_processing = topn.run_formula_assignment(bg_path, bg_scans, cfg)
                bg_tables = assignment_tables(bg_spectrum, bg_classified)
                for label, table in [("sample", sample_unambiguous),
                                     ("background", bg_tables["unambiguous"])]:
                    if not np.isfinite(pd.to_numeric(table["Peak Height"], errors="coerce")).all():
                        raise ValueError(f"Nonfinite {label} intensities before background removal")
                bg_config = BackgroundConfig(enabled=True, removal_mode=options["background_removal_mode"],
                                             sample_to_background_ratio_cutoff=options["sample_background_ratio_cutoff"])
                after_background, audit, removal = topn.remove_paired_background_formulas(
                    sample_unambiguous, bg_tables["unambiguous"], bg_config)
                removal = {key: value for key, value in removal.items() if key != "intensity_subtraction_performed"}
                removal["background_removal_performed"] = True

                after_internal_standard, internal_standard_removed, internal_standard_report = (
                    remove_internal_standard_peaks(
                        after_background,
                        options["internal_standard_formulas"],
                        options["remove_internal_standard_peaks"],
                    )
                )
                formulas = after_internal_standard["Molecular Formula"]
                if formulas.isna().any() or formulas.astype(str).str.strip().eq("").any():
                    raise ValueError("Level 3 contains an unambiguous peak without a molecular formula")
                has_oxygen = formulas.map(
                    lambda formula: formula_element_counts(formula).get("O", 0) > 0
                )
                oxygen_removed = after_internal_standard.loc[~has_oxygen].copy()
                after_oxygen = after_internal_standard.loc[has_oxygen].copy()
                sample_clean = clean_dbe_o_assignments(after_oxygen, options["gap_settings"])
                final = sample_clean["unambiguous"]
                save_gap_diagnostics(sample_clean, plots, "03_sample_after_internal_standard")

                public_details = write_public_level3(
                    nc, final, sample_result["tables"], formula_element_counts
                )
                nc.background_removal_performed = 1
                nc.internal_standard_removal_applied = int(options["remove_internal_standard_peaks"])
                nc.n_internal_standard_peaks_removed = internal_standard_report["n_removed_peaks"]
                nc.oxygen_filter_applied = 1
                nc.dbe_o_filter_applied = 1
                render_level3_van_krevelen(
                    nc, final, plots / "07_level3_van_krevelen.png"
                )

                diagnostic_path = diagnostics_dir / "level3_audit.nc"
                def write_level3_audit(audit_nc):
                    group = audit_nc.createGroup("level3")
                    bg_group = group.createGroup("background")
                    export_tables(bg_group, {"selected_scans": background, **bg_tables})
                    nc_text(bg_group, "processing_metadata_json", json_text(bg_processing))
                    export_tables(group.createGroup("sample_chemical_cleaning"), sample_clean)
                    export_tables(group, {
                        "background_candidates": background_candidates,
                        "input_sample_unambiguous": sample_unambiguous,
                        "unambiguous_after_background": after_background,
                        "internal_standard_removed": internal_standard_removed,
                        "unambiguous_after_internal_standard": after_internal_standard,
                        "oxygen_removed": oxygen_removed,
                        "unambiguous_after_oxygen": after_oxygen,
                        "unambiguous_after_dbe_o": final,
                        "removal_audit": audit,
                    })
                    nc_text(group, "processing_metadata_json", json_text({
                        "gap_settings": options["gap_settings"],
                        "background_removal": removal,
                        "internal_standard_removal": internal_standard_report,
                    }))
                write_diagnostic_level(diagnostic_path, "level3", write_level3_audit)
                state["diagnostic_files"]["level3"] = str(diagnostic_path)
                return {
                    "n_unambiguous_before_cleaning": int(sample_unambiguous["m/z"].nunique()),
                    "n_unambiguous_after_background": int(after_background["m/z"].nunique()),
                    "n_removed_internal_standards": internal_standard_report["n_removed_peaks"],
                    "n_unambiguous_after_internal_standard": int(after_internal_standard["m/z"].nunique()),
                    "internal_standard_removal_applied": options["remove_internal_standard_peaks"],
                    "n_removed_without_oxygen": int(oxygen_removed["m/z"].nunique()),
                    "n_unambiguous_after_oxygen": int(after_oxygen["m/z"].nunique()),
                    "n_unambiguous_after_dbe_o": public_details["n_final_peaks"],
                    "n_retained_unambiguous": public_details["n_final_peaks"],
                    "background_removal_performed": True,
                    **public_details,
                }


            complete("level3", level3_writer)
            update_completed_sample_metadata(state, sample_metadata, data_dir)
        elif "n_background_scans_averaged" in sample_metadata:
            sample_metadata["n_background_scans_averaged"] = 0
            update_completed_sample_metadata(state, sample_metadata, data_dir)
        state["status"] = "success"
        save_json(state_path, state)
        return state
    except BaseException as exc:
        state.update(status="failed", error=f"{type(exc).__name__}: {exc}", failed_level=active_level)
        if active_level and state["levels"][active_level]["status"] != "success":
            state["levels"][active_level] = {"status": "failed", "error": state["error"],
                                               "background_removal_performed": False}
        save_json(state_path, state)
        print("Run failed; completed files remain available. Status:", state_path)
        raise


# Reader-facing named NetCDF arrays (public schema v1).
"""Reader-facing NetCDF arrays for the three-level CoreMS workflow.

The full processing and audit tables are kept separately.  The functions
here take its in-memory outputs and write a small, named-variable data product.
No RAW processing, formula assignment, or scientific filtering happens here.
"""


from collections.abc import Callable, Mapping

import numpy as np
import pandas as pd


PUBLIC_SCHEMA_VERSION = "corems-three-levels-public-1.0"
ELEMENTS = ("C", "H", "O", "N", "S")
ASSIGNMENT_STATES = ("unambiguous", "ambiguous", "unassigned")


def _require_columns(frame: pd.DataFrame, columns: tuple[str, ...]) -> None:
    absent = [name for name in columns if name not in frame.columns]
    if absent:
        raise ValueError(f"CoreMS table is missing required columns: {absent}")


def _numeric(series: pd.Series, name: str, *, allow_missing: bool = False) -> np.ndarray:
    numbers = pd.to_numeric(series, errors="coerce").to_numpy(dtype=np.float64)
    if (not allow_missing and not np.isfinite(numbers).all()) or np.isinf(numbers).any():
        raise ValueError(f"Invalid {name} in CoreMS table")
    return numbers


def _first_observed_peaks(rows: pd.DataFrame) -> pd.DataFrame:
    """Build one measured-peak record per exact observed m/z, preserving order."""
    _require_columns(rows, ("m/z", "Peak Height", "assign_type"))
    if rows.empty:
        return pd.DataFrame(columns=["m/z", "Peak Height", "assign_type", "peak_id"])
    mz = _numeric(rows["m/z"], "observed m/z")
    heights = _numeric(rows["Peak Height"], "Peak Height")
    if (mz <= 0).any() or (heights < 0).any():
        raise ValueError("Expected positive observed m/z and nonnegative Peak Height")
    if not rows["assign_type"].isin(ASSIGNMENT_STATES).all():
        raise ValueError("Unknown CoreMS assignment status")

    # Several candidate formulas may belong to the *same* measured peak.
    # They must never create several peak intensities or separate peak IDs.
    for observed_mz, group in rows.groupby("m/z", sort=False):
        if group["assign_type"].nunique(dropna=False) != 1:
            raise ValueError(f"Conflicting assignment statuses for m/z {observed_mz}")
        measured = _numeric(group["Peak Height"], "Peak Height")
        if not np.allclose(measured, measured[0], rtol=1e-9, atol=1e-9):
            raise ValueError(f"Conflicting measured peak intensities for m/z {observed_mz}")
    peaks = rows.drop_duplicates("m/z", keep="first")[
        ["m/z", "Peak Height", "assign_type"]
    ].reset_index(drop=True).copy()
    peaks["peak_id"] = np.arange(len(peaks), dtype=np.int64)
    return peaks


def _element_arrays(formulas: pd.Series, element_counts: Callable[[str], Mapping[str, int]]) -> dict[str, np.ndarray]:
    parsed: list[Mapping[str, int]] = []
    for value in formulas:
        if pd.isna(value) or not str(value).strip():
            raise ValueError("An assigned formula is empty")
        atoms = element_counts(str(value))
        if not atoms:
            raise ValueError(f"Unable to read assigned formula: {value!r}")
        unsupported = set(atoms) - set(ELEMENTS)
        if unsupported:
            raise ValueError(f"Formula {value!r} contains unsupported elements: {sorted(unsupported)}")
        parsed.append(atoms)
    result: dict[str, np.ndarray] = {}
    for element in ELEMENTS:
        values = [atoms.get(element, 0) for atoms in parsed]
        if any(not isinstance(x, (int, np.integer)) or x < 0 for x in values):
            raise ValueError(f"Invalid {element} atom count")
        result[element] = np.asarray(values, dtype=np.int32)
    return result


def _normalize_column_name(value: object) -> str:
    return "".join(c for c in str(value).lower() if c.isalnum())


def _mass_errors(
    assigned: pd.DataFrame, *, explicit_column: str | None = None
) -> tuple[np.ndarray, str, str]:
    """Use only a declared ppm field or an already provided theoretical m/z.

    In particular, do not calculate theoretical m/z from a molecular formula:
    that would require additional charge/adduct/isotope assumptions.
    """
    if explicit_column is not None and explicit_column not in assigned.columns:
        raise ValueError(f"Requested m/z error column {explicit_column!r} is absent")

    ppm_names = (
        "mzerrorppm", "masserrorppm", "mzppmerror", "ppmerror",
        "massppmerror", "errorppm", "massaccuracyppm", "mzdeviationppm",
    )
    ppm_column = explicit_column
    if ppm_column is None:
        ranked = sorted(
            (
                (ppm_names.index(normalized), position, column)
                for position, column in enumerate(assigned.columns)
                if (normalized := _normalize_column_name(column)) in ppm_names
            ),
            key=lambda item: (item[0], item[1]),
        )
        if ranked:
            ppm_column = ranked[0][2]
    if ppm_column is not None:
        values = _numeric(assigned[ppm_column], str(ppm_column), allow_missing=True)
        return values, f"CoreMS dataframe column: {ppm_column}", "CoreMS values, sign as provided"

    theoretical_names = (
        "calculatedmz", "mzcalculated", "mzcalc", "calcmz",
        "theoreticalmz", "mztheoretical", "mztheor",
    )
    ranked = sorted(
        (
            (theoretical_names.index(normalized), position, column)
            for position, column in enumerate(assigned.columns)
            if (normalized := _normalize_column_name(column)) in theoretical_names
        ),
        key=lambda item: (item[0], item[1]),
    )
    if ranked:
        calc_column = ranked[0][2]
        calculated = _numeric(assigned[calc_column], str(calc_column), allow_missing=True)
        observed = _numeric(assigned["m/z"], "observed m/z")
        with np.errstate(divide="ignore", invalid="ignore"):
            ppm = (observed - calculated) / calculated * 1e6
        ppm[~np.isfinite(ppm)] = np.nan
        return ppm, f"Derived from CoreMS dataframe column: {calc_column}", "(observed m/z - calculated m/z) / calculated m/z * 1e6"

    return (
        np.full(len(assigned), np.nan, dtype=np.float64),
        "unavailable in CoreMS dataframe",
        "not calculated; no ppm error or theoretical m/z column",
    )


def _collapse_duplicate_formulas(
    assigned: pd.DataFrame, *, mz_error_column: str | None = None
) -> tuple[pd.DataFrame, np.ndarray, str, str]:
    """One candidate per formula and measured peak; preserve full rows in audits.

    Repeated records for one candidate must agree on error.  Silently taking a
    different ppm error from an arbitrary duplicate would alter the result.
    """
    errors, source, convention = _mass_errors(assigned, explicit_column=mz_error_column)
    for (_, formula), indexes in assigned.groupby(
        ["m/z", "Molecular Formula"], sort=False
    ).indices.items():
        values = errors[indexes]
        if not np.allclose(values, values[0], rtol=0, atol=1e-12, equal_nan=True):
            raise ValueError(f"Conflicting m/z errors for duplicate formula {formula!r}")
    keep = ~assigned.duplicated(["m/z", "Molecular Formula"], keep="first")
    return assigned.loc[keep].copy(), errors[keep.to_numpy()], source, convention


def prepare_level2(
    tables: Mapping[str, pd.DataFrame],
    element_counts: Callable[[str], Mapping[str, int]],
    *,
    mz_error_column: str | None = None,
) -> tuple[dict[str, np.ndarray], dict[str, str | int]]:
    """Separate observed peaks from their zero/one/multiple candidate formulas."""
    rows = tables["assignments_all"]
    _require_columns(rows, ("Molecular Formula",))
    peaks = _first_observed_peaks(rows)
    mapping = pd.Series(peaks["peak_id"].to_numpy(), index=peaks["m/z"])
    assigned = rows.loc[rows["assign_type"].ne("unassigned")].copy()
    if assigned["Molecular Formula"].isna().any():
        raise ValueError("Assigned CoreMS row has no molecular formula")
    assigned, errors, error_source, convention = _collapse_duplicate_formulas(
        assigned, mz_error_column=mz_error_column
    )
    _require_columns(rows, ("Calibrated m/z",))
    calibrated_mz = _numeric(rows.drop_duplicates("m/z", keep="first")["Calibrated m/z"], "Calibrated m/z")
    _require_columns(assigned, ("Calculated m/z",))
    calculated_mz = _numeric(assigned["Calculated m/z"], "Calculated m/z")
    # if (calculated_mz <= 0).any():
    #     raise ValueError("Invalid calculated m/z in Level 2 assignments")
    n_candidates = assigned.groupby("m/z", sort=False)["Molecular Formula"].nunique()
    peak_counts = peaks["m/z"].map(n_candidates).fillna(0).astype(int)
    if ((peaks["assign_type"].eq("unassigned") & peak_counts.ne(0)) |
        (peaks["assign_type"].eq("unambiguous") & peak_counts.ne(1)) |
        (peaks["assign_type"].eq("ambiguous") & peak_counts.le(1))).any():
        raise ValueError("CoreMS assignment status does not match candidate formula counts")
    data = {
        "peak_id": peaks["peak_id"].to_numpy(dtype=np.int64),
        "peak_mz": peaks["m/z"].to_numpy(dtype=np.float64),
        "calibrated_mz": calibrated_mz,
        "peak_intensity": peaks["Peak Height"].to_numpy(dtype=np.float64),
        "assignment_status": peaks["assign_type"].astype(str).to_numpy(dtype=object),
        "formula_peak_id": assigned["m/z"].map(mapping).to_numpy(dtype=np.int64),
        "formula": assigned["Molecular Formula"].astype(str).to_numpy(dtype=object),
        "calculated_mz": calculated_mz,
        "mz_error_ppm": errors,
        **_element_arrays(assigned["Molecular Formula"], element_counts),
    }
    details = {
        "n_total_peaks": len(peaks),
        "n_unambiguous_peaks": int(peaks["assign_type"].eq("unambiguous").sum()),
        "n_ambiguous_peaks": int(peaks["assign_type"].eq("ambiguous").sum()),
        "n_unassigned_peaks": int(peaks["assign_type"].eq("unassigned").sum()),
        "n_formula_rows": len(assigned),
        "mz_error_ppm_source": error_source,
        "mz_error_ppm_convention": convention,
        "n_missing_assigned_mz_errors": int(np.isnan(errors).sum()),
    }
    return data, details


def prepare_level3(
    final: pd.DataFrame,
    level2_tables: Mapping[str, pd.DataFrame],
    element_counts: Callable[[str], Mapping[str, int]],
    *,
    mz_error_column: str | None = None,
) -> tuple[dict[str, np.ndarray], dict[str, str | int]]:
    """Represent final unambiguous formula assignments as a stick spectrum."""
    original_peaks = _first_observed_peaks(level2_tables["assignments_all"])
    mapping = pd.Series(original_peaks["peak_id"].to_numpy(), index=original_peaks["m/z"])
    _require_columns(final, ("m/z", "Peak Height", "assign_type", "Molecular Formula"))
    if not final["assign_type"].eq("unambiguous").all():
        raise ValueError("Level 3 must only contain Level 2 unambiguous formulas")
    _first_observed_peaks(final)
    if final["Molecular Formula"].isna().any():
        raise ValueError("Level 3 contains missing formulas")
    final, errors, error_source, convention = _collapse_duplicate_formulas(
        final, mz_error_column=mz_error_column
    )
    _require_columns(final, ("Calibrated m/z", "Calculated m/z"))
    calibrated_mz = _numeric(final["Calibrated m/z"], "Calibrated m/z")
    calculated_mz = _numeric(final["Calculated m/z"], "Calculated m/z")
    
    counts = _element_arrays(final["Molecular Formula"], element_counts)
    if (counts["O"] < 1).any():
        raise ValueError("Level 3 contains a formula with no oxygen")
    if final["m/z"].duplicated().any():
        raise ValueError("Level 3 must have one final formula per measured m/z")
    source = final["m/z"].map(mapping)
    if source.isna().any():
        raise ValueError("Level 3 includes an m/z that is absent from Level 2")
    source_ids = source.to_numpy(dtype=np.int64)
    source_formulas = level2_tables["assignments_all"][["m/z", "Molecular Formula"]].drop_duplicates()
    allowed_pairs = set(map(tuple, source_formulas.to_numpy(dtype=object)))
    if any(pair not in allowed_pairs for pair in final[["m/z", "Molecular Formula"]].itertuples(index=False, name=None)):
        raise ValueError("Level 3 includes a formula that was absent from Level 2")
    original_status = original_peaks.set_index("peak_id")["assign_type"]
    if not original_status.iloc[source_ids].eq("unambiguous").all():
        raise ValueError("Level 3 includes a peak that was not Level 2 unambiguous")
    original_heights = original_peaks.set_index("peak_id")["Peak Height"].iloc[source_ids].to_numpy(dtype=float)
    current_heights = _numeric(final["Peak Height"], "Level 3 Peak Height")
    if not np.allclose(original_heights, current_heights, rtol=1e-9, atol=1e-9):
        raise ValueError("Level 3 must retain the measured Level 2 sample intensity")
    data = {
        "source_peak_id": source_ids,
        "peak_mz": final["m/z"].to_numpy(dtype=np.float64),
        "calibrated_mz": calibrated_mz,
        "peak_intensity": current_heights,
        "formula": final["Molecular Formula"].astype(str).to_numpy(dtype=object),
        "calculated_mz": calculated_mz, 
        "mz_error_ppm": errors,
        **counts,
    }
    details = {
        "n_final_peaks": len(final),
        "mz_error_ppm_source": error_source,
        "mz_error_ppm_convention": convention,
        "n_missing_assigned_mz_errors": int(np.isnan(errors).sum()),
    }
    return data, details


def _dimension(nc, name: str, size: int) -> None:
    # An unlimited dimension permits a legitimate empty assignment table.
    nc.createDimension(name, size if size else None)


def _variable(nc, name: str, dtype: str | type, dimension: str, values: np.ndarray, *, units: str = "", description: str = "") -> None:
    options = {} if dtype is str else {"zlib": True, "complevel": 4}
    var = nc.createVariable(name, dtype, (dimension,), **options)
    if len(values):
        var[:] = values
    if units:
        var.units = units
    if description:
        var.description = description


def _metadata(nc, level: str) -> None:
    nc.file_level_code = {"level1": "a1", "level2": "b1", "level3": "c1"}[level]
    nc.public_schema_version = PUBLIC_SCHEMA_VERSION


def write_public_level1(nc, snapshot: Mapping[str, object], scans: list[int]) -> dict[str, int]:
    mz = np.asarray(snapshot["profile_mz"], dtype=np.float64)
    intensity = np.asarray(snapshot["profile_intensity"], dtype=np.float64)
    peaks = snapshot["peaks"]
    if not isinstance(peaks, pd.DataFrame):
        raise TypeError("snapshot['peaks'] must be a pandas DataFrame")
    _require_columns(peaks, ("mz", "intensity", "resolving_power", "snr"))
    if not len(mz) or len(mz) != len(intensity) or not np.isfinite(mz).all() or not np.isfinite(intensity).all() or (np.diff(mz) < 0).any():
        raise ValueError("Invalid averaged profile")
    _metadata(nc, "level1")
    _dimension(nc, "profile_point", len(mz))
    _dimension(nc, "peak", len(peaks))
    _variable(nc, "profile_mz", "f8", "profile_point", mz, units="m/z")
    _variable(nc, "profile_intensity", "f8", "profile_point", intensity, units="instrument abundance")
    for name, column, units in (
        ("peak_mz", "mz", "m/z"),
        ("peak_intensity", "intensity", "instrument abundance"),
        ("peak_resolving_power", "resolving_power", "dimensionless"),
        ("peak_snr", "snr", "dimensionless"),
    ):
        values = _numeric(peaks[column], column, allow_missing=column in {"resolving_power", "snr"})
        _variable(nc, name, "f8", "peak", values, units=units)
    nc.n_scans_averaged = len(scans)
    return {"n_scans_averaged": len(scans), "n_peaks": len(peaks)}


def write_public_level2(
    nc, tables: Mapping[str, pd.DataFrame], element_counts: Callable[[str], Mapping[str, int]],
    *, mz_error_column: str | None = None,
) -> dict[str, str | int]:
    data, details = prepare_level2(tables, element_counts, mz_error_column=mz_error_column)
    _metadata(nc, "level2")
    _dimension(nc, "peak", len(data["peak_id"]))
    _dimension(nc, "formula_row", len(data["formula"]))
    _variable(nc, "peak_id", "i8", "peak", data["peak_id"], description="Zero-based ID of one measured peak")
    _variable(nc, "peak_mz", "f8", "peak", data["peak_mz"], units="m/z")
    _variable(nc, "peak_intensity", "f8", "peak", data["peak_intensity"], units="instrument abundance")
    _variable(nc, "assignment_status", str, "peak", data["assignment_status"])
    _variable(nc, "formula_peak_id", "i8", "formula_row", data["formula_peak_id"], description="Links to peak_id; zero, one, or multiple candidate formulas per peak")
    _variable(nc, "formula", str, "formula_row", data["formula"])
    _variable(nc, "calibrated_mz", "f8", "peak", data["calibrated_mz"], units="m/z")
    nc.variables["calibrated_mz"].source = "CoreMS dataframe column: Calibrated m/z"
    _variable(nc, "calculated_mz", "f8", "formula_row", data["calculated_mz"], units="m/z")
    nc.variables["calculated_mz"].source = "CoreMS dataframe column: Calculated m/z"
    _variable(nc, "mz_error_ppm", "f8", "formula_row", data["mz_error_ppm"], units="ppm")
    for name in ELEMENTS:
        _variable(nc, name, "i4", "formula_row", data[name], units="atoms")
    nc.variables["mz_error_ppm"].source = details["mz_error_ppm_source"]
    nc.variables["mz_error_ppm"].convention = details["mz_error_ppm_convention"]
    for name in ("n_total_peaks", "n_unambiguous_peaks", "n_ambiguous_peaks", "n_unassigned_peaks", "n_formula_rows"):
        setattr(nc, name, details[name])
    return details


def write_public_level3(
    nc, final: pd.DataFrame, level2_tables: Mapping[str, pd.DataFrame],
    element_counts: Callable[[str], Mapping[str, int]], *, mz_error_column: str | None = None,
) -> dict[str, str | int]:
    data, details = prepare_level3(final, level2_tables, element_counts, mz_error_column=mz_error_column)
    _metadata(nc, "level3")
    _dimension(nc, "peak", len(data["peak_mz"]))
    _variable(nc, "source_peak_id", "i8", "peak", data["source_peak_id"], description="Corresponding zero-based b1 peak_id")
    _variable(nc, "peak_mz", "f8", "peak", data["peak_mz"], units="m/z")
    _variable(nc, "peak_intensity", "f8", "peak", data["peak_intensity"], units="original averaged sample instrument abundance")
    _variable(nc, "formula", str, "peak", data["formula"])
    _variable(nc, "calibrated_mz", "f8", "peak", data["calibrated_mz"], units="m/z")
    _variable(nc, "calculated_mz", "f8", "peak", data["calculated_mz"], units="m/z")
    nc.variables["calibrated_mz"].source = "CoreMS dataframe column: Calibrated m/z"
    nc.variables["calculated_mz"].source = "CoreMS dataframe column: Calculated m/z"
    _variable(nc, "mz_error_ppm", "f8", "peak", data["mz_error_ppm"], units="ppm")
    for name in ELEMENTS:
        _variable(nc, name, "i4", "peak", data[name], units="atoms")
    nc.variables["mz_error_ppm"].source = details["mz_error_ppm_source"]
    nc.variables["mz_error_ppm"].convention = details["mz_error_ppm_convention"]
    nc.n_final_peaks = details["n_final_peaks"]
    nc.background_intensity_subtracted = 0
    return details


def validate_public_level(path, level: str) -> dict[str, bool]:
    """Read only named arrays; enforce joins and final formula constraints."""
    from netCDF4 import Dataset

    if level not in {"level1", "level2", "level3"}:
        raise ValueError(f"Unsupported public level: {level}")
    with Dataset(path) as nc:
        if nc.public_schema_version != PUBLIC_SCHEMA_VERSION or public_output_level(nc) != level:
            raise AssertionError("Wrong public data product version or level")
        if nc.groups or any(name.startswith("c000") for name in nc.variables):
            raise AssertionError("The public file must not contain internal audit tables")
        required = {"peak_mz", "peak_intensity"}
        if level == "level1":
            required |= {"profile_mz", "profile_intensity", "peak_resolving_power", "peak_snr"}
        elif level == "level2":
            required |= {"peak_id", "assignment_status", "formula_peak_id", "formula", "calculated_mz", "calibrated_mz", "mz_error_ppm", *ELEMENTS}
        else:
            required |= {"source_peak_id", "formula", "calculated_mz", "calibrated_mz", "mz_error_ppm", *ELEMENTS}
        absent = required - set(nc.variables)
        if absent:
            raise AssertionError(f"Missing named public arrays: {sorted(absent)}")
        n_peaks = len(nc.variables["peak_mz"])
        if len(nc.variables["peak_intensity"]) != n_peaks:
            raise AssertionError("Peak intensities are not aligned")
        if level == "level1":
            if len(nc.variables["profile_mz"]) != len(nc.variables["profile_intensity"]):
                raise AssertionError("Level 1 profile arrays are not aligned")
            if any(len(nc.variables[name]) != n_peaks for name in ("peak_resolving_power", "peak_snr")):
                raise AssertionError("Level 1 picked peak arrays are not aligned")
        elif level == "level2":
            ids = np.asarray(nc.variables["peak_id"][:], dtype=np.int64)
            if not np.array_equal(ids, np.arange(n_peaks, dtype=np.int64)):
                raise AssertionError("Invalid Level 2 peak IDs")
            if len(nc.variables["calibrated_mz"]) != n_peaks:
                raise AssertionError("Level 2 calibrated m/z is not aligned with peaks")
            n_formula_rows = len(nc.variables["formula"])
            fields = ("formula_peak_id", "calculated_mz", "mz_error_ppm", *ELEMENTS)
            if any(len(nc.variables[name]) != n_formula_rows for name in fields):
                raise AssertionError("Formula candidate arrays are not aligned")
            parent = np.asarray(nc.variables["formula_peak_id"][:], dtype=np.int64)
            if ((parent < 0) | (parent >= n_peaks)).any():
                raise AssertionError("A candidate formula references a missing peak")
            status = np.asarray(nc.variables["assignment_status"][:], dtype=str)
            if len(status) != n_peaks or not set(status).issubset(ASSIGNMENT_STATES):
                raise AssertionError("Invalid Level 2 assignment states")
            formula_values = np.asarray(nc.variables["formula"][:], dtype=str)
            attached_unique = np.asarray([
                len(set(formula_values[parent == i])) for i in range(n_peaks)
            ])
            if ((status == "unassigned") & (attached_unique != 0)).any() or ((status == "unambiguous") & (attached_unique != 1)).any() or ((status == "ambiguous") & (attached_unique < 2)).any():
                raise AssertionError("Assignment states do not match candidate formula links")
        else:
            aligned = ("source_peak_id", "calibrated_mz", "formula", "calculated_mz", "mz_error_ppm", *ELEMENTS)
            if any(len(nc.variables[name]) != n_peaks for name in aligned):
                raise AssertionError("Level 3 formula arrays are not aligned")
            source = np.asarray(nc.variables["source_peak_id"][:], dtype=np.int64)
            if (source < 0).any() or len(np.unique(source)) != len(source):
                raise AssertionError("Invalid Level 2 source peak references")
            if (np.asarray(nc.variables["O"][:], dtype=np.int32) < 1).any():
                raise AssertionError("Level 3 contains an oxygen-free formula")
    return {"validated": True}


## 4. Run one RAW file

Every run gets a new directory. 
Inspect `three_level_status.json` to distinguish skipped c1 from a processing failure.

In [ ]:
options = {
    "modules": CANDIDATE_MODULES, "tracer_priority": TRACER_PRIORITY,
    "top_n_sample": TOP_N_SAMPLE_SCANS, "sample_random_seed": SAMPLE_RANDOM_SEED,
    "ms_filter": MS_FILTER, "eic_tolerance_ppm": EIC_TOLERANCE_PPM,
    "roi_settings": ROI_SETTINGS, "gap_settings": GAP_SETTINGS,
    "background_mode": BACKGROUND_MODE, "background_raw_path": BACKGROUND_RAW_PATH,
    "top_n_background": TOP_N_BACKGROUND_SCANS, "background_seed": BACKGROUND_RANDOM_SEED,
    "background_removal_mode": BACKGROUND_REMOVAL_MODE,
    "sample_background_ratio_cutoff": SAMPLE_TO_BACKGROUND_RATIO_CUTOFF,
    "remove_internal_standard_peaks": REMOVE_INTERNAL_STANDARD_PEAKS,
    "internal_standard_formulas": INTERNAL_STANDARD_FORMULAS,
    "metadata_fields": METADATA_FIELDS,
    "field_metadata": FIELD_METADATA,
    "new_file_name": NEW_FILE_NAME,
}
result = run_three_level_workflow(RAW_FILE, OUTPUT_ROOT, corems_config, options)
RUN_DIR = Path(result["run_directory"])
NETCDF_FILES = {level: Path(path) for level, path in result["netcdf_files"].items()}
display(pd.DataFrame(result["levels"]).T)
print("Run directory:", RUN_DIR)

## 5. Output map and public variables

| File | Contents |
|---|---|
| `data/<prefix>.a1.<date>.<time>.nc` | `profile_mz`, `profile_intensity` over `profile_point`; `peak_mz`, `peak_intensity`, `peak_resolving_power`, `peak_snr` over `peak` |
| `data/<prefix>.b1.<date>.<time>.nc` | Measured `peak_id`, `peak_mz`, `calibrated_mz`, `peak_intensity`, `assignment_status` over `peak`; `formula_peak_id`, `formula`, `calculated_mz`, `mz_error_ppm`, `C`, `H`, `O`, `N`, `S` over `formula_row` |
| `data/<prefix>.c1.<date>.<time>.nc` | Final stick spectrum: `source_peak_id`, `peak_mz`, `calibrated_mz`, `peak_intensity`, `formula`, `calculated_mz`, `mz_error_ppm`, `C`, `H`, `O`, `N`, `S` over `peak`; embedded `van_krevelen_png` over `van_krevelen_png_byte` |
| `data/raw_metadata.json` | Sample acquisition fields, selected tracer, sample/background scan average counts |
| `data/background_raw_metadata.json` | External background acquisition fields, when applicable |
| `diagnostics/` | Scan selection, full CoreMS assignments, background, internal-standard, oxygen, and DBE-O audits, processing configuration/status, and plots |

The formula arrays in b1 have **one row per candidate formula**. An unassigned peak has no formula row, while an ambiguous peak links to multiple formula rows. Peak intensities are stored once per measured peak. C/H/O/N/S are explicit atom counts. Levels 2 and 3 are derived from the same sample average in a1; they do not duplicate the complete profile. A c1 plot of retained peaks is a stick spectrum, not a background-subtracted continuous profile.
When enabled, `diagnostics/level3_audit.nc` records `internal_standard_removed` and `unambiguous_after_internal_standard` under `level3`. `data/<prefix>.c1.<date>.<time>.nc` records the applied flag and removed-peak count as attributes.
